# EXP-012 LLM-primary labels + Axial fold0

唯一变量: labels005(报告规则) → **LLM-primary**(steven v4_blend)。
- gold 58 例优先覆盖;非 gold 用 LLM soft
- 静音桶 0.25/0.5 → NaN(不进损失);再 clip 到 [0.08,0.92]
- 对照: Axial labels005 **0.699** / S+C+A labels005 **0.721**
- 预期: LLM key≈0.89 > 规则 key≈0.81,同架构应抬分


In [ ]:
# 提交模式:禁网(enable_internet:false),torch 2.2.0+cu118 + nvidia-cu11 依赖全部离线安装
# 坑1:Kaggle 挂载把文件名中 '+' 去掉(2.2.0cu118=非法 PEP440)→ 复制为合法名
# 坑2:torch 2.2.0+cu118 的 CUDA 库在 nvidia-*-cu11 依赖包中(不在 torch wheel 内),
#     只装 torch 会缺 libcudart(v8/v9 实证)→ 必须显式安装全部 13 个 wheel
# 坑3:/kaggle/working 会被收集为 kernel 输出 → 装完立即删除
import subprocess, sys, os, glob, shutil
wh = sorted(glob.glob('/kaggle/input/**/*.whl', recursive=True))
print('挂载 wheel 共', len(wh), '个')
for w in wh: print('  ', os.path.basename(w))
assert len(wh) >= 13, 'FAIL: 离线 wheel 不足(应 13: torch+torchvision+11 nvidia-cu11)'
fixed = []
for w in wh:
    base = os.path.basename(w)
    dst = os.path.join('/kaggle/working', base.replace('cu118', '+cu118', 1) if 'cu118' in base else base)
    shutil.copy(w, dst)
    fixed.append(dst)
print('安装 wheel:', [os.path.basename(f) for f in fixed])
subprocess.check_call([sys.executable,"-m","pip","uninstall","-y","-q","torch","torchvision","torchaudio"])
# 显式安装全部 wheel(--no-deps --no-index: 无版本解析/无联网)
subprocess.check_call([sys.executable,"-m","pip","install","-q","--no-deps","--no-index"] + fixed)
for f in fixed: os.remove(f)
import numpy, torch, torchvision, sklearn, pandas
print("torch", torch.__version__, "cuda", torch.version.cuda, "| numpy", numpy.__version__)
assert torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# preflight: competition mount + Axial coverage
# Correct layout: test_series/<StudyInstanceUID>/<SeriesInstanceUID>/*.dcm
import os, csv
from collections import defaultdict

base = '/kaggle/input'

def _find(b, depth=0, max_depth=3):
    if not os.path.isdir(b):
        return None
    try:
        entries = sorted(os.listdir(b))
    except OSError:
        return None
    for e in entries:
        p = os.path.join(b, e)
        if not os.path.isdir(p):
            continue
        if e in ('train_series', 'test_series'):
            continue
        if os.path.isfile(os.path.join(p, 'test.csv')) and os.path.isdir(os.path.join(p, 'test_series')):
            return p
        if depth < max_depth - 1:
            r = _find(p, depth + 1)
            if r:
                return r
    return None

root = _find(base)
if root is None:
    raise SystemExit('FAIL: 未找到含 test_series+test.csv 的比赛挂载')
print('root =', root)

def count_dcm(dirpath):
    if not os.path.isdir(dirpath):
        return -1
    return sum(1 for f in os.listdir(dirpath) if f.lower().endswith('.dcm'))

def plane_stats(series_csv, series_dir, label):
    if not os.path.isfile(series_csv):
        print(label, 'series csv missing', series_csv)
        return
    with open(series_csv, newline='', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    if not rows:
        raise SystemExit(f'FAIL: {label} series csv empty')
    plane_key = None
    for k in rows[0].keys():
        if 'plane' in k.lower() or 'anatom' in k.lower():
            plane_key = k
            break
    uid_key = 'StudyInstanceUID'
    ser_key = 'SeriesInstanceUID'
    series_counts = defaultdict(int)
    study_sets = defaultdict(set)
    axial_series_rows = []
    for r in rows:
        pl = r.get(plane_key, '?') if plane_key else '?'
        series_counts[pl] += 1
        if uid_key in r:
            study_sets[pl].add(r[uid_key])
        if pl == 'Axial' and ser_key in r:
            axial_series_rows.append(r)
    print(label, 'plane series counts', dict(series_counts))
    print(label, 'plane study counts', {k: len(v) for k, v in study_sets.items()})
    print(label, 'Axial studies', len(study_sets.get('Axial', set())))
    assert len(study_sets.get('Axial', set())) > 0, f'FAIL: {label} 无 Axial study'

    # Walk both possible layouts
    n_dcm = 0
    hits = 0
    misses = []
    for r in axial_series_rows:
        suid = r[uid_key]
        seid = r[ser_key]
        candidates = [
            os.path.join(series_dir, suid, seid),
            os.path.join(series_dir, seid),
            os.path.join(series_dir, suid, 'series', seid),
        ]
        found = None
        for c in candidates:
            k = count_dcm(c)
            if k and k > 0:
                found = c
                n_dcm += k
                hits += 1
                print(f'  axial OK study={suid[:14]} series={seid[:16]} dcm={k} path={c}')
                break
        if found is None:
            # fallback: list study dir
            sdir = os.path.join(series_dir, suid)
            if os.path.isdir(sdir):
                children = sorted(os.listdir(sdir))[:8]
                print(f'  axial miss study={suid[:14]} series={seid[:16]} study_children={children}')
            else:
                print(f'  axial miss study={suid[:14]} series={seid[:16]} NO_STUDY_DIR {sdir}')
            misses.append(seid)
    print(label, f'axial series hits={hits}/{len(axial_series_rows)} dcm_total={n_dcm}')
    assert hits > 0 and n_dcm > 0, f'FAIL: {label} Axial DICOM 为空 (hits={hits}, dcm={n_dcm})'

    # also sample sagittal for sanity
    sag = [r for r in rows if (r.get(plane_key) if plane_key else '') == 'Sagittal']
    if sag:
        r0 = sag[0]
        for c in [os.path.join(series_dir, r0[uid_key], r0[ser_key]), os.path.join(series_dir, r0[ser_key])]:
            k = count_dcm(c)
            if k and k > 0:
                print(label, 'sagittal sanity', r0[ser_key][:16], 'dcm', k)
                break

plane_stats(os.path.join(root, 'test_series.csv'), os.path.join(root, 'test_series'), 'test')
plane_stats(os.path.join(root, 'train_series.csv'), os.path.join(root, 'train_series'), 'train')
print('preflight axial: OK')


In [ ]:
import base64, os
# embed rsna_v3.py
parts=[]
parts.append("IyAtKi0gY29kaW5nOiB1dGYtOCAtKi0NCiIiIg0KUlNOQSBLbmVlIEFibm9ybWFsaXR5IERldGVjdGlvbiDigJQgdjIo5oyJIEFuYXRvbWljYWxfUGxhbmUg5YiG5bmz6Z2i54us56uL5bu65qihICsg6J6N5ZCIKQ0KPT09PT09PT09PT09PT09PT09PT09PT09PT09")
parts.append("PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0NCuebuOWvuSBiYXNlbGluZShyc25hX2Jhc2VsaW5lLnB5KeeahOaguOW/g+aUueWKqDoNCi0gYmFzZWxpbmUg5oqKIHN0dWR5IOWGheaJgOacieW5s+mdoueahCBzZXJp")
parts.append("ZXMg57KX5pq05ou85oiQ6YCa6YGTO3YyIOaMiSBBbmF0b21pY2FsX1BsYW5lDQogIChBeGlhbCAvIFNhZ2l0dGFsIC8gQ29yb25hbCnlkITorq3kuIDkuKrni6znq4sgUmVzTmV0MzQs5bmz6Z2i5YaF6YOo5YaN5YGaIDIuNUQgY2xpcDsNCi0gdGVzdCDmjqjnkIbm")
parts.append("l7bmr4/kuKrlubPpnaLmqKHlnovni6znq4vpooTmtYss5pyA57uI5oyJIOWQhOW5s+mdoiBPT0Yg5a6P5bmz5Z2HIEFVQyDliqDmnYPono3lkIgNCiAgKOadg+mHjeWPryAtLWZ1c2Ugd2VpZ2h0IOeUqCBBVUMg5Yqg5p2DLOaIliAtLWZ1c2UgbWVhbiDnroDljZXl")
parts.append("ubPlnYcpOw0KLSDmn5Agc3R1ZHkg57y65p+Q5bmz6Z2i5pe2LOiejeWQiOWPquWvueivpSBzdHVkeSDlrp7pmYXlj6/nlKjnmoTlubPpnaLlvZLkuIDljJbmnYPph43jgIINCg0K5YW25L2Z6YC76L6RKOi9r+agh+etvuWhq+WFheOAgUdyb3VwS0ZvbGTjgIF2YWxp")
parts.append("ZCDmjqnnoIHmjZ/lpLHjgIFESUNPTSDpooTlpITnkIYp5LiOIGJhc2VsaW5lIOS4gOiHtOOAgg0KDQrov5DooYznjq/looM6S2FnZ2xlIE5vdGVib29rKEdQVSzmjILovb3mr5TotZvmlbDmja4gKyDnpL7ljLrova/moIfnrb7mlbDmja7pm4Yp44CCDQrnlKjms5U6")
parts.append("DQogICAgIXB5dGhvbiByc25hX3YyLnB5IC0tZXBvY2hzIDMgLS1iYXRjaCAxNiAtLW4tY2xpcHMgMiAtLXBsYW5lcyBzYWdpdHRhbCxjb3JvbmFsDQogICAgIXB5dGhvbiByc25hX3YyLnB5IC0tZXBvY2hzIDMgLS1iYXRjaCAxNiAtLWZ1c2Ugd2VpZ2h0IC0tcHNl")
parts.append("dWRvLWNvbmYgMC4zDQrovpPlh7o6IHN1Ym1pc3Npb24uY3N2KFN0dWR5SW5zdGFuY2VVSUQgKyAxMiDliJfnva7kv6HliIbmlbApDQoiIiINCmltcG9ydCBhcmdwYXJzZQ0KaW1wb3J0IGpzb24NCmltcG9ydCBvcw0KaW1wb3J0IHN5cw0KaW1wb3J0IHRpbWUNCmlt")
parts.append("cG9ydCB3YXJuaW5ncw0KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoDQoNCmltcG9ydCBudW1weSBhcyBucA0KaW1wb3J0IHBhbmRhcyBhcyBwZA0KDQp0cnk6DQogICAgaW1wb3J0IHRvcmNoDQogICAgaW1wb3J0IHRvcmNoLm5uIGFzIG5uDQogICAgZnJvbSB0b3Jj")
parts.append("aC51dGlscy5kYXRhIGltcG9ydCBEYXRhc2V0LCBEYXRhTG9hZGVyDQogICAgZnJvbSB0b3JjaC5vcHRpbSBpbXBvcnQgQWRhbVcNCiAgICBmcm9tIHRvcmNoLm9wdGltLmxyX3NjaGVkdWxlciBpbXBvcnQgQ29zaW5lQW5uZWFsaW5nTFINCiAgICBIQVNfVE9SQ0gg")
parts.append("PSBUcnVlDQpleGNlcHQgSW1wb3J0RXJyb3I6DQogICAgSEFTX1RPUkNIID0gRmFsc2UNCg0KdHJ5Og0KICAgIGltcG9ydCB0b3JjaHZpc2lvbi5tb2RlbHMgYXMgdHZtDQogICAgSEFTX1RWID0gVHJ1ZQ0KZXhjZXB0IEltcG9ydEVycm9yOg0KICAgIEhBU19UViA9")
parts.append("IEZhbHNlDQoNCnRyeToNCiAgICBpbXBvcnQgcHlkaWNvbQ0KICAgIEhBU19QWURJQ09NID0gVHJ1ZQ0KZXhjZXB0IEltcG9ydEVycm9yOg0KICAgIEhBU19QWURJQ09NID0gRmFsc2UNCg0KdHJ5Og0KICAgIGltcG9ydCByc25hX3JlcG9ydF9sYWJlbHMgYXMgUkwg")
parts.append("ICAjIOaKpeWRiuKGkjEyIOeXheeBtuW8seagh+etvijkuI4gcnNuYV92My5weSDlkIznm67lvZUpDQogICAgSEFTX1JMID0gVHJ1ZQ0KZXhjZXB0IEltcG9ydEVycm9yOg0KICAgIEhBU19STCA9IEZhbHNlDQoNCmZyb20gc2tsZWFybi5tb2RlbF9zZWxlY3Rpb24g")
parts.append("aW1wb3J0IEdyb3VwS0ZvbGQNCmZyb20gc2tsZWFybi5tZXRyaWNzIGltcG9ydCByb2NfYXVjX3Njb3JlDQoNCndhcm5pbmdzLmZpbHRlcndhcm5pbmdzKCJpZ25vcmUiKQ0KDQpMQUJFTFMgPSBbIkFDTCIsICJNQ0wiLCAiTWVkaWFsIE1lbmlzY3VzIiwgIkxhdGVy")
parts.append("YWwgTWVuaXNjdXMiLA0KICAgICAgICAgICJNZWRpYWwgT0EiLCAiTGF0ZXJhbCBPQSIsICJQRiBPQSIsICJFZmZ1c2lvbiIsICJTeW5vdml0aXMiLA0KICAgICAgICAgICJCYWtlcidzIiwgIkNvbnR1c2lvbiIsICJGcmFjdHVyZSJdDQpMQUJFTFNfRU4gPSBbIkFD")
parts.append("TCIsICJNQ0wiLCAiTWVkaWFsTWVuaXNjdXMiLCAiTGF0ZXJhbE1lbmlzY3VzIiwNCiAgICAgICAgICAgICAiTWVkaWFsT0EiLCAiTGF0ZXJhbE9BIiwgIlBGT0EiLCAiRWZmdXNpb24iLCAiU3lub3ZpdGlzIiwNCiAgICAgICAgICAgICAiQmFrZXJzIiwgIkNvbnR1")
parts.append("c2lvbiIsICJGcmFjdHVyZSJdICAjIOWIl+WQjeWuieWFqOWMlijlhpnmj5DkuqTnlKgpDQpQTEFORVMgPSBbIkF4aWFsIiwgIlNhZ2l0dGFsIiwgIkNvcm9uYWwiXQ0KDQpJTUdfU0laRSA9IDIyNA0KQ0xJUCA9IDMgICAgICAgICAgIyDmr48gY2xpcCDliIfniYfm")
parts.append("lbAocHJldi9jZW50ZXIvbmV4dCkNCk1BWF9TRVJJRVMgPSA0ICAgICMg5q+PIHN0dWR5IOavj+S4quW5s+mdouacgOWkmuWPlueahCBzZXJpZXMg5pWwKHYyIOaMieW5s+mdouWPlizljZXlubPpnaIgc2VyaWVzIOabtOWwkSw0IOi2s+WknykNCkZPTERTID0gNQ0K")
parts.append("U0VFRCA9IDQyDQoNCg0KZGVmIHJlc29sdmVfcGF0aHMoKToNCiAgICAiIiLmlbDmja7lrprkvY3pobrluo86MSkgL2thZ2dsZS9pbnB1dCDmjILovb07Mikga2FnZ2xlaHViIOiHquWKqOS4i+i9vTszKSDmnKzlnLAgcnNuYV9rbmVlLyDnm67lvZXjgIIiIiINCiAg")
parts.append("ICAjIDEpIEthZ2dsZSDmjILovb0o5YW85a655aSa5bGC5oyC6L29Oi9rYWdnbGUvaW5wdXQvPOaVsOaNruWQjT4vIOaIliAva2FnZ2xlL2lucHV0LzzliIbnsbs+LzzmlbDmja7lkI0+LykNCiAgICBpZiBvcy5wYXRoLmlzZGlyKCIva2FnZ2xlL2lucHV0Iik6DQog")
parts.append("ICAgICAgIGZvciBzdWIgaW4gc29ydGVkKG9zLmxpc3RkaXIoIi9rYWdnbGUvaW5wdXQiKSk6DQogICAgICAgICAgICBkID0gb3MucGF0aC5qb2luKCIva2FnZ2xlL2lucHV0Iiwgc3ViKQ0KICAgICAgICAgICAgZm9yIGNhbmQgaW4gKGQsIG9zLnBhdGguam9pbihk")
parts.append("LCBzdWIpKTogICMg5aSE55CGIEthZ2dsZSDlj4zlsYLmjILovb0NCiAgICAgICAgICAgICAgICBpZiBvcy5wYXRoLmV4aXN0cyhvcy5wYXRoLmpvaW4oY2FuZCwgInRyYWluLmNzdiIpKToNCiAgICAgICAgICAgICAgICAgICAgcmV0dXJuIGNhbmQsICIva2FnZ2xl")
parts.append("L3dvcmtpbmciDQogICAgICAgICAgICBpZiBvcy5wYXRoLmlzZGlyKGQpOiAgIyDlho3lvoDkuIvkuIDlsYLmib4o5aaCIC9rYWdnbGUvaW5wdXQvY29tcGV0aXRpb25zLzxuYW1lPi8pDQogICAgICAgICAgICAgICAgZm9yIHN1YjIgaW4gc29ydGVkKG9zLmxpc3Rk")
parts.append("aXIoZCkpOg0KICAgICAgICAgICAgICAgICAgICBjYW5kID0gb3MucGF0aC5qb2luKGQsIHN1YjIpDQogICAgICAgICAgICAgICAgICAgIGlmIG9zLnBhdGguZXhpc3RzKG9zLnBhdGguam9pbihjYW5kLCAidHJhaW4uY3N2IikpOg0KICAgICAgICAgICAgICAgICAg")
parts.append("ICAgICAgcmV0dXJuIGNhbmQsICIva2FnZ2xlL3dvcmtpbmciDQogICAgICAgIHByaW50KCJb6K+K5patXSAva2FnZ2xlL2lucHV0IOWtmOWcqCzkvYbmsqHmib7liLDlkKsgdHJhaW4uY3N2IOeahOebruW9leOAguWunumZheaMgui9veWGheWuueWmguS4izoiLCBm")
parts.append("bHVzaD1UcnVlKQ0KICAgICAgICBmb3Igc3ViIGluIHNvcnRlZChvcy5saXN0ZGlyKCIva2FnZ2xlL2lucHV0IikpOg0KICAgICAgICAgICAgcCA9IG9zLnBhdGguam9pbigiL2thZ2dsZS9pbnB1dCIsIHN1YikNCiAgICAgICAgICAgIHRyeToNCiAgICAgICAgICAg")
parts.append("ICAgICBpbm5lciA9IG9zLmxpc3RkaXIocCkNCiAgICAgICAgICAgIGV4Y2VwdCBPU0Vycm9yOg0KICAgICAgICAgICAgICAgIGlubmVyID0gWyIo5peg5rOV6K+75Y+WKSJdDQogICAgICAgICAgICBwcmludChmIiAgL2thZ2dsZS9pbnB1dC97c3VifS8gIC0+ICB7")
parts.append("aW5uZXJbOjhdfSIsIGZsdXNoPVRydWUpDQogICAgICAgIHByaW50KCJb6K+K5patXSDlpoLmnpzkvaDmjILnmoTmmK8gJ+S9nOiAheaVsOaNruWMhS/liKvkurrnmoQga2VybmVsIOi+k+WHuicsIOmHjOmdouayoeaciSB0cmFpbi5jc3Yg5ZKMIHRyYWluX3Nlcmll")
parts.append("cy8sIiwNCiAgICAgICAgICAgICAgIuW/hemhu+aMguavlOi1m+aVsOaNruacrOS9kyByc25hLWtuZWUtYWJub3JtYWxpdHktZGV0ZWN0aW9uIChBZGQgSW5wdXQgLT4gQ29tcGV0aXRpb25zIOagh+etvumhtSkiLCBmbHVzaD1UcnVlKQ0KICAgICMgMikga2FnZ2xl")
parts.append("aHViIOWFnOW6lTrmnKrmjILovb3ml7boh6rliqjkuIvovb3mr5TotZvmlbDmja4oS2FnZ2xlIE5vdGVib29rIOWGhee9ruWHreaNrikNCiAgICB0cnk6DQogICAgICAgIGltcG9ydCBrYWdnbGVodWINCiAgICAgICAgZCA9IGthZ2dsZWh1Yi5jb21wZXRpdGlvbl9k")
parts.append("b3dubG9hZCgicnNuYS1rbmVlLWFibm9ybWFsaXR5LWRldGVjdGlvbiIpDQogICAgICAgIGlmIG9zLnBhdGguZXhpc3RzKG9zLnBhdGguam9pbihkLCAidHJhaW4uY3N2IikpOg0KICAgICAgICAgICAgcHJpbnQoZiJb6Lev5b6EXSDmnKrmjILovb0s5bey55SoIGth")
parts.append("Z2dsZWh1YiDojrflj5bmlbDmja46IHtkfSIsIGZsdXNoPVRydWUpDQogICAgICAgICAgICByZXR1cm4gZCwgIi9rYWdnbGUvd29ya2luZyINCiAgICBleGNlcHQgSW1wb3J0RXJyb3I6DQogICAgICAgIHBhc3MgICMg5pys5Zyw5pyq6KOFIGthZ2dsZWh1Yg0KICAg")
parts.append("IGV4Y2VwdCBFeGNlcHRpb24gYXMgZToNCiAgICAgICAgcHJpbnQoZiJb5o+Q56S6XSBrYWdnbGVodWIg5LiL6L295pWw5o2u5aSx6LSlOiB7ZX0iLCBmbHVzaD1UcnVlKQ0KICAgICMgMykg5pys5Zyw5Zue6YCAKOiEmuacrOebruW9lSAvIGN3ZCAvIGN3ZCDkuIvn")
parts.append("moQgcnNuYV9rbmVlIOWtkOebruW9lemDveaJvikNCiAgICBjYW5kcyA9IFtdDQogICAgdHJ5Og0KICAgICAgICBjYW5kcy5hcHBlbmQob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguYWJzcGF0aChfX2ZpbGVfXykpKQ0KICAgIGV4Y2VwdCBOYW1lRXJyb3I6DQogICAg")
parts.append("ICAgIHBhc3MNCiAgICBjYW5kcy5hcHBlbmQob3MuZ2V0Y3dkKCkpDQogICAgZm9yIGJhc2UgaW4gbGlzdChjYW5kcyk6DQogICAgICAgIGNhbmRzLmFwcGVuZChvcy5wYXRoLmpvaW4oYmFzZSwgInJzbmFfa25lZSIpKQ0KICAgIHNlZW4gPSBzZXQoKQ0KICAgIGZv")
parts.append("ciBiYXNlIGluIGNhbmRzOg0KICAgICAgICBpZiBiYXNlIGluIHNlZW46DQogICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICBzZWVuLmFkZChiYXNlKQ0KICAgICAgICBpZiBvcy5wYXRoLmV4aXN0cyhvcy5wYXRoLmpvaW4oYmFzZSwgInRyYWluLmNzdiIpKToN")
parts.append("CiAgICAgICAgICAgIHJldHVybiBiYXNlLCBiYXNlDQogICAgc3lzLmV4aXQoIuaJvuS4jeWIsOaVsOaNrjror7flhYjlnKjmr5TotZvpobXliqDlhaXmr5TotZvlubblkIzmhI/mlbDmja7op4TliJks5YaNIEFkZCBJbnB1dCDmjILovb0gIg0KICAgICAgICAgICAg")
parts.append("ICJyc25hLWtuZWUtYWJub3JtYWxpdHktZGV0ZWN0aW9uO+acrOWcsOivt+WcqCByc25hX2tuZWUvIOS4i+i/kOihjOOAgiIpDQoNCg0KX1BTRVVET19GSUxFUyA9ICgidHJhaW5fZm9sZHNfd2l0aF9wc2V1ZG8uY3N2IiwgInRyYWluX3dpdGhfcHNldWRvLmNzdiIs")
parts.append("DQogICAgICAgICAgICAgICAgICJ0cmFpbl93aXRoX3NvZnRfbGFiZWxzLmNzdiIsICJ0cmFpbl9wc2V1ZG8uY3N2IikNCl9TS0lQX0RJUlMgPSB7InRyYWluX3NlcmllcyIsICJ0ZXN0X3NlcmllcyJ9ICAjIERJQ09NIOWkp+ebruW9lSznu53kuI3ljrvpgY3ljoYN")
parts.append("Cl9XRUlHSFRfRVhUID0gKCIucHRoIiwgIi5wdCIsICIuY2twdCIsICIucHRoLnRhciIpICAjIOemu+e6v+mihOiuree7g+adg+mHjeaWh+S7tuaJqeWxleWQjQ0KDQojIDEyIOagh+etvuWkmuivreiogOWFs+mUruivjeivjeWFuCjoi7Ev6KW/L+iNty/lvrcv5rOV")
parts.append("KeKAlOKAlOmbtuaMgui9vei9r+agh+etvuWFnOW6lQ0KIyDlkb3kuK0g4oaSIOmrmOe9ruS/oSzmnKrlkb3kuK0g4oaSIOS9jue9ruS/oSjova/moIfnrb4s5LiN57uZ56GsIDAvMSzpgb/lhY3plJnor43or6/lr7wpDQpfUkVQT1JUX0tFWVdPUkRTID0gew0KICAg")
parts.append("ICJBQ0wiOiBbImFjbCIsICJhbnRlcmlvciBjcnVjaWF0ZSIsICJjcnVjaWFkbyBhbnRlcmlvciIsICJ2b29yc3RlIGtydWlzYmFuZCIsDQogICAgICAgICAgICAidm9yZGVyZW4ga3JldXpiYW5kIiwgImxjYSIsICJsaWdhbWVudCBjcm9pc8OpIGFudMOpcmlldXIi")
parts.append("XSwNCiAgICAiTUNMIjogWyJtY2wiLCAibWVkaWFsIGNvbGxhdGVyYWwiLCAiY29sYXRlcmFsIG1lZGlhbCIsICJtZWRpYWxlIGNvbGxhdGVyYWxlIiwNCiAgICAgICAgICAgICJpbm5lcmVzIHNlaXRlbmJhbmQiLCAibGlnYW1lbnQgY29sbGF0w6lyYWwgbcOpZGlh")
parts.append("bCJdLA0KICAgICJNZWRpYWwgTWVuaXNjdXMiOiBbIm1lZGlhbCBtZW5pc2N1cyIsICJtZW5pc2NvIGludGVybm8iLCAibWVuaXNjdXMgbWVkaWFsaXMiLA0KICAgICAgICAgICAgICAgICAgICAgICAgIm1lbmlza3VzIG1lZGlhbCIsICJtZW5pc3F1ZSBtw6lkaWFs")
parts.append("IiwgImlubmVyIG1lbmlzY3VzIiwgIm1lbmlzY28gbWVkaWFsIl0sDQogICAgIkxhdGVyYWwgTWVuaXNjdXMiOiBbImxhdGVyYWwgbWVuaXNjdXMiLCAibWVuaXNjbyBleHRlcm5vIiwgIm1lbmlzY3VzIGxhdGVyYWxpcyIsDQogICAgICAgICAgICAgICAgICAgICAg")
parts.append("ICAgIm1lbmlza3VzIGxhdGVyYWwiLCAibWVuaXNxdWUgbGF0w6lyYWwiLCAib3V0ZXIgbWVuaXNjdXMiXSwNCiAgICAiTWVkaWFsIE9BIjogWyJtZWRpYWwgb3N0ZW9hcnRocml0aXMiLCAiYXJ0cm9zaXMgZmVtb3JvdGliaWFsIG1lZGlhbCIsDQogICAgICAgICAg")
parts.append("ICAgICAgICAibWVkaWFsZSBnb25hcnRyb3NlIiwgIm1lZGlhbGUgYXJ0aHJvc2UiLCAib3N0ZW9hcnRyaXRpcyBtZWRpYWwiLA0KICAgICAgICAgICAgICAgICAgIm1lZGlhbCBjb21wYXJ0bWVudCBuYXJyb3dpbmciLCAibWVkaWFsIGpvaW50IHNwYWNlIG5hcnJv")
parts.append("d2luZyJdLA0KICAgICJMYXRlcmFsIE9BIjogWyJsYXRlcmFsIG9zdGVvYXJ0aHJpdGlzIiwgImFydHJvc2lzIGZlbW9yb3RpYmlhbCBsYXRlcmFsIiwNCiAgICAgICAgICAgICAgICAgICAibGF0ZXJhbGUgZ29uYXJ0cm9zZSIsICJsYXRlcmFsZSBhcnRocm9zZSIs")
parts.append("ICJsYXRlcmFsIGpvaW50IHNwYWNlIG5hcnJvd2luZyJdLA0KICAgICJQRiBPQSI6IFsicGF0ZWxsb2ZlbW9yYWwgb3N0ZW9hcnRocml0aXMiLCAiYXJ0cm9zaXMgcGF0ZWxvZmVtb3JhbCIsDQogICAgICAgICAgICAgICJwYXRlbGxvZmVtb3JhbGUgYXJ0cm9zZSIs")
parts.append("ICJwYXRlbGxvZmVtb3JhbCBhcnRocm9zZSIsICJwZiBvYSIsDQogICAgICAgICAgICAgICJwYXRlbGxvZmVtb3JhbCBuYXJyb3dpbmciLCAicGF0ZWxsb2ZlbW9yYWwgam9pbnQgc3BhY2UiXSwNCiAgICAiRWZmdXNpb24iOiBbImVmZnVzaW9uIiwgImRlcnJhbWUi")
parts.append("LCAidm9jaHQiLCAiZXJndXNzIiwgIsOpcGFuY2hlbWVudCIsICJqb2ludCBlZmZ1c2lvbiJdLA0KICAgICJTeW5vdml0aXMiOiBbInN5bm92aXRpcyIsICJzaW5vdml0aXMiLCAic3lub3ZpYWxlIiwgInN5bm92aWFsIHRoaWNrZW5pbmciLCAic3lub3ZpdGlzIl0s")
parts.append("DQogICAgIkJha2VyJ3MiOiBbImJha2VyIiwgInBvcGxpdGVhbCBjeXN0IiwgInF1aXN0ZSBkZSBiYWtlciIsICJiYWtlciBjeXN0Il0sDQogICAgIkNvbnR1c2lvbiI6IFsiY29udHVzaW9uIiwgImJvbmUgYnJ1aXNlIiwgImNvbnR1c2nDs24iLCAia25ldXppbmci")
parts.append("LCAia29udHVzaW9uIiwNCiAgICAgICAgICAgICAgICAgICJtYXJyb3cgZWRlbWEiLCAiw7NzZW8iLCAiYm9uZSBicnVpc2luZyJdLA0KICAgICJGcmFjdHVyZSI6IFsiZnJhY3R1cmUiLCAiZnJhY3R1cmEiLCAiZnJhY3R1dXIiLCAiZnJha3R1ciIsICJmcmFjdHVy")
parts.append("ZSBsaW5lIiwNCiAgICAgICAgICAgICAgICAgImZyYWN0dXJlIGxpbmUiLCAic3ViY2hvbmRyYWwgZnJhY3R1cmUiXSwNCn0NCg0KDQpkZWYgX3BzZXVkb19mcm9tX3JlcG9ydHModHJhaW5fY3N2LCBjb25mX2hpdD0wLjksIGNvbmZfbWlzcz0wLjEpOg0KICAgICIi")
parts.append("IuS7juaUvuWwhOaKpeWRiuaWh+acrOeUqOWkmuivreiogOWFs+mUruivjeeUn+aIkCAxMiDmoIfnrb7ova/moIfnrb4o6Zu25oyC6L295YWc5bqVKeOAgg0KDQogICAg6L+U5ZueIERhdGFGcmFtZShTdHVkeUluc3RhbmNlVUlEICsgcHNldWRvXyog5YiXKeOAguWF")
parts.append("s+mUruivjeWRveS4reKGkmNvbmZfaGl0LA0KICAgIOacquWRveS4reKGkmNvbmZfbWlzcyjkuI3nu5nnoawgMC8xLOmZjeS9jumUmeivjeivr+WvvCnjgILlrpjmlrnlt7LmnInmoIfms6jnmoQgc3R1ZHkNCiAgICDku43ku6XlrpjmlrnkuLrlh4YobG9hZF9sYWJl")
parts.append("bHMg55qE5aGr5YWF6YC76L6R5L+d6K+B5a6Y5pa55LyY5YWIKeOAgg0KICAgICIiIg0KICAgIGlmICJSZXBvcnQiIG5vdCBpbiB0cmFpbl9jc3YuY29sdW1uczoNCiAgICAgICAgcmV0dXJuIE5vbmUNCiAgICByZXAgPSB0cmFpbl9jc3ZbIlJlcG9ydCJdLmZpbGxu")
parts.append("YSgiIikNCiAgICByb3dzID0ge30NCiAgICBmb3IgbGFiLCBrd3MgaW4gX1JFUE9SVF9LRVlXT1JEUy5pdGVtcygpOg0KICAgICAgICBsb3cgPSBbay5sb3dlcigpIGZvciBrIGluIGt3c10NCiAgICAgICAgcm93c1tmInBzZXVkb197bGFifSJdID0gcmVwLmFwcGx5")
parts.append("KA0KICAgICAgICAgICAgbGFtYmRhIHQ6IGNvbmZfaGl0IGlmIGFueShrIGluIHN0cih0KS5sb3dlcigpIGZvciBrIGluIGxvdykgZWxzZSBjb25mX21pc3MpDQogICAgZGYgPSBwZC5EYXRhRnJhbWUoeyJTdHVkeUluc3RhbmNlVUlEIjogdHJhaW5fY3N2WyJTdHVk")
parts.append("eUluc3RhbmNlVUlEIl0sICoqcm93c30pDQogICAgcmV0dXJuIGRmDQoNCg0KZGVmIF9wcmludF9tb3VudF90cmVlKGJhc2U9Ii9rYWdnbGUvaW5wdXQiLCB0YWc9IuaVsOaNriIsIG1heF9maWxlcz02KToNCiAgICAiIiLmiZPljbDmjILovb3nm67lvZXmoJEo5pyA")
parts.append("5aSa5Lik5bGCLOavj+ebruW9leWJjeWHoOS4quaWh+S7tiks5biu55So5oi356Gu6K6k5oyC6L295YaF5a6544CCIiIiDQogICAgaWYgbm90IG9zLnBhdGguaXNkaXIoYmFzZSk6DQogICAgICAgIHByaW50KGYiW+iviuaWrS17dGFnfV0g55uu5b2V5LiN5a2Y5Zyo")
parts.append("OiB7YmFzZX0iLCBmbHVzaD1UcnVlKQ0KICAgICAgICByZXR1cm4NCiAgICBwcmludChmIlvor4rmlq0te3RhZ31dIC9rYWdnbGUvaW5wdXQg5a6e6ZmF5oyC6L295YaF5a65OiIsIGZsdXNoPVRydWUpDQogICAgZm9yIHN1YiBpbiBzb3J0ZWQob3MubGlzdGRpcihi")
parts.append("YXNlKSk6DQogICAgICAgIHAgPSBvcy5wYXRoLmpvaW4oYmFzZSwgc3ViKQ0KICAgICAgICB0cnk6DQogICAgICAgICAgICBpZiBvcy5wYXRoLmlzZGlyKHApOg0KICAgICAgICAgICAgICAgIGlubmVyID0gc29ydGVkKG9zLmxpc3RkaXIocCkpWzptYXhfZmlsZXNd")
parts.append("DQogICAgICAgICAgICAgICAgcHJpbnQoZiIgIHtzdWJ9LyAtPiB7aW5uZXJ9IiwgZmx1c2g9VHJ1ZSkNCiAgICAgICAgICAgIGVsc2U6DQogICAgICAgICAgICAgICAgcHJpbnQoZiIgIHtzdWJ9ICjmlofku7YpIiwgZmx1c2g9VHJ1ZSkNCiAgICAgICAgZXhjZXB0")
parts.append("IE9TRXJyb3I6DQogICAgICAgICAgICBwcmludChmIiAge3N1Yn0gKOaXoOazleivu+WPlikiLCBmbHVzaD1UcnVlKQ0KDQoNCmRlZiBfcHJpbnRfZGNtX3RyZWUocm9vdCwgdGFnPSJESUNPTSIsIG1heF9zaG93PTUpOg0KICAgICIiIuaJk+WNsCB0cmFpbl9zZXJp")
parts.append("ZXMg55uu5b2V5YmN5Yeg5bGC55yf5a6e57uT5p6EICsg5oq95qC3IC5kY20g6K6h5pWwLOehruiupCBESUNPTSDluIPlsYDjgIINCg0KICAgIFJTTkEg5bi46KeB5biD5bGAOnRyYWluX3Nlcmllcy88U3R1ZHlJbnN0YW5jZVVJRD4vPFNlcmllc0luc3RhbmNlVUlE")
parts.append("Pi8qLmRjbSjkuKTlsYIpOw0KICAgIOS5n+WFvOWuueS4gOWxgig8U2VyaWVzSW5zdGFuY2VVSUQ+LyouZGNtKeaIluabtOa3seWxgizov5nph4zmiZPljbDlrp7pmYXnnIvliLDnmoTjgIINCiAgICAiIiINCiAgICBkY20gPSBvcy5wYXRoLmpvaW4ocm9vdCwgInRy")
parts.append("YWluX3NlcmllcyIpDQogICAgaWYgbm90IG9zLnBhdGguaXNkaXIoZGNtKToNCiAgICAgICAgcHJpbnQoZiJb6K+K5patLXt0YWd9XSB7ZGNtfSDkuI3lrZjlnKghIiwgZmx1c2g9VHJ1ZSkNCiAgICAgICAgcmV0dXJuDQogICAgdHJ5Og0KICAgICAgICBzdWJzX2Fs")
parts.append("bCA9IHNvcnRlZChvcy5saXN0ZGlyKGRjbSkpDQogICAgZXhjZXB0IE9TRXJyb3I6DQogICAgICAgIHN1YnNfYWxsID0gW10NCiAgICBzdWJzID0gc3Vic19hbGxbOm1heF9zaG93XQ0KICAgIHByaW50KGYiW+iviuaWrS17dGFnfV0ge2RjbX0iLCBmbHVzaD1UcnVl")
parts.append("KQ0KICAgIHByaW50KGYiW+iviuaWrS17dGFnfV0g5a2Q55uu5b2VL+aWh+S7tuaVsDoge2xlbihzdWJzX2FsbCl9LOagt+acrDoiLCBmbHVzaD1UcnVlKQ0KICAgIGZvciBzIGluIHN1YnM6DQogICAgICAgIHNwID0gb3MucGF0aC5qb2luKGRjbSwgcykNCiAgICAg")
parts.append("ICAgaWYgb3MucGF0aC5pc2RpcihzcCk6DQogICAgICAgICAgICB0cnk6DQogICAgICAgICAgICAgICAgaW5uZXIgPSBzb3J0ZWQob3MubGlzdGRpcihzcCkpWzptYXhfc2hvd10NCiAgICAgICAgICAgIGV4Y2VwdCBPU0Vycm9yOg0KICAgICAgICAgICAgICAgIGlu")
parts.append("bmVyID0gW10NCiAgICAgICAgICAgIHByaW50KGYiICB7c30vIC0+IHtpbm5lcn0iLCBmbHVzaD1UcnVlKQ0KICAgICAgICAgICAgZm9yIGl0IGluIGlubmVyWzoyXToNCiAgICAgICAgICAgICAgICBpdHAgPSBvcy5wYXRoLmpvaW4oc3AsIGl0KQ0KICAgICAgICAg")
parts.append("ICAgICAgIGlmIG9zLnBhdGguaXNkaXIoaXRwKToNCiAgICAgICAgICAgICAgICAgICAgdHJ5Og0KICAgICAgICAgICAgICAgICAgICAgICAgZmlsZXMgPSBzb3J0ZWQob3MubGlzdGRpcihpdHApKVs6M10NCiAgICAgICAgICAgICAgICAgICAgZXhjZXB0IE9TRXJy")
parts.append("b3I6DQogICAgICAgICAgICAgICAgICAgICAgICBmaWxlcyA9IFtdDQogICAgICAgICAgICAgICAgICAgIHByaW50KGYiICAgICAge2l0fS8gLT4ge2ZpbGVzfSIsIGZsdXNoPVRydWUpDQogICAgICAgIGVsc2U6DQogICAgICAgICAgICBwcmludChmIiAge3N9ICjm")
parts.append("lofku7YpIiwgZmx1c2g9VHJ1ZSkNCiAgICAjIOaKveagtyAuZGNtIOiuoeaVsDrliY0gMyBzdHVkeSDDlyDliY0gMyBzZXJpZXMNCiAgICBuX2RjbSA9IDANCiAgICB0cnk6DQogICAgICAgIGZvciBzIGluIHN1YnNfYWxsWzozXToNCiAgICAgICAgICAgIHNwID0g")
parts.append("b3MucGF0aC5qb2luKGRjbSwgcykNCiAgICAgICAgICAgIGlmIG5vdCBvcy5wYXRoLmlzZGlyKHNwKToNCiAgICAgICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICAgICAgZm9yIGl0IGluIHNvcnRlZChvcy5saXN0ZGlyKHNwKSlbOjNdOg0KICAgICAgICAgICAg")
parts.append("ICAgIGl0cCA9IG9zLnBhdGguam9pbihzcCwgaXQpDQogICAgICAgICAgICAgICAgaWYgb3MucGF0aC5pc2RpcihpdHApOg0KICAgICAgICAgICAgICAgICAgICBuX2RjbSArPSBzdW0oMSBmb3IgZiBpbiBvcy5saXN0ZGlyKGl0cCkgaWYgZi5sb3dlcigpLmVuZHN3")
parts.append("aXRoKCIuZGNtIikpDQogICAgZXhjZXB0IE9TRXJyb3I6DQogICAgICAgIHBhc3MNCiAgICBwcmludChmIlvor4rmlq0te3RhZ31dIOaKveagtyAuZGNtIOaWh+S7tuaVsDoge25fZGNtfSjliY0gMyBzdHVkeSDDlyDliY0gMyBzZXJpZXMpIiwgZmx1c2g9VHJ1ZSkN")
parts.append("Cg0KDQpkZWYgX3NjYW5fb2ZmbGluZV93ZWlnaHRzKGJhc2U9Ii9rYWdnbGUvaW5wdXQiLCBkZXB0aD0wLCBtYXhfZGVwdGg9NCk6DQogICAgIiIi6YCS5b2S5omr5o+P5oyC6L2955uu5b2V5LiL55qE6aKE6K6t57uD5p2D6YeN5paH5Lu2KC5wdGgvLnB0IOetiSks")
parts.append("6Lez6L+HIERJQ09NIOWkp+ebruW9leOAgg0KDQogICAg6L+U5ZueIFso57ud5a+56Lev5b6ELCDnm7jlr7kgL2thZ2dsZS9pbnB1dCDnmoTmj4/ov7ApXTvmib7kuI3liLDov5Tlm57nqbrliJfooajjgIINCiAgICDnpL7ljLrmiZPljIXnmoTnprvnur/mnYPph43m")
parts.append("lbDmja7pm4Yo5aaCIG1hdHRlb25hY2NhcmF0by9yZXNuZXQzNC1pbWFnZW5ldC1wdGjjgIENCiAgICBtYXJjc2hhZGUvcmVzbmV0LXByZXRyYWluZWQtd2VpZ2h0cy1pbWFnZW5ldOOAgXB5dG9yY2gvcmVzbmV0MzQp5oyC6L295ZCO5Y2z5Y+v6KKr5omr5YiwLA0K")
parts.append("ICAgIOaXoOmcgOiBlOe9keOAgg0KICAgICIiIg0KICAgIGhpdHMgPSBbXQ0KICAgIGlmIGRlcHRoID4gbWF4X2RlcHRoIG9yIG5vdCBvcy5wYXRoLmlzZGlyKGJhc2UpOg0KICAgICAgICByZXR1cm4gaGl0cw0KICAgIHRyeToNCiAgICAgICAgZW50cmllcyA9IHNv")
parts.append("cnRlZChvcy5saXN0ZGlyKGJhc2UpKQ0KICAgIGV4Y2VwdCBPU0Vycm9yOg0KICAgICAgICByZXR1cm4gaGl0cw0KICAgIGZvciBmbiBpbiBlbnRyaWVzOg0KICAgICAgICBwID0gb3MucGF0aC5qb2luKGJhc2UsIGZuKQ0KICAgICAgICBpZiBvcy5wYXRoLmlzZmls")
parts.append("ZShwKSBhbmQgZm4ubG93ZXIoKS5lbmRzd2l0aChfV0VJR0hUX0VYVCk6DQogICAgICAgICAgICBoaXRzLmFwcGVuZCgocCwgb3MucGF0aC5yZWxwYXRoKHAsICIva2FnZ2xlL2lucHV0IikpKQ0KICAgICAgICBlbGlmIG9zLnBhdGguaXNkaXIocCkgYW5kIGZuIG5v")
parts.append("dCBpbiBfU0tJUF9ESVJTOg0KICAgICAgICAgICAgaGl0cy5leHRlbmQoX3NjYW5fb2ZmbGluZV93ZWlnaHRzKHAsIGRlcHRoICsgMSwgbWF4X2RlcHRoKSkNCiAgICByZXR1cm4gaGl0cw0KDQoNCmRlZiBfbG9hZF9zdGF0ZV9kaWN0X2Zyb21fZmlsZShwYXRoKToN")
parts.append("CiAgICAiIiLku47mnYPph43mlofku7bor7sgc3RhdGVfZGljdCjlhbzlrrk655u05o6lIHN0YXRlX2RpY3QgLyDljIXlnKggY2hlY2twb2ludCDph4wgLyDlkITnp43liY3nvIAp44CCDQoNCiAgICDor7vlj5blpLHotKXmiJbmoLzlvI/kuI3lr7nov5Tlm54gTm9u")
parts.append("ZeOAgg0KICAgICIiIg0KICAgIHRyeToNCiAgICAgICAgb2JqID0gdG9yY2gubG9hZChwYXRoLCBtYXBfbG9jYXRpb249ImNwdSIsIHdlaWdodHNfb25seT1UcnVlKQ0KICAgIGV4Y2VwdCBUeXBlRXJyb3I6ICAjIOaXp+eJiCB0b3JjaCDml6Agd2VpZ2h0c19vbmx5")
parts.append("IOWPguaVsA0KICAgICAgICBvYmogPSB0b3JjaC5sb2FkKHBhdGgsIG1hcF9sb2NhdGlvbj0iY3B1IikNCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6DQogICAgICAgIHByaW50KGYiW+adg+mHjV0ge3BhdGh9IOivu+WPluWksei0pToge2V9IiwgZmx1c2g9VHJ1")
parts.append("ZSkNCiAgICAgICAgcmV0dXJuIE5vbmUNCiAgICBpZiBpc2luc3RhbmNlKG9iaiwgZGljdCkgYW5kICJzdGF0ZV9kaWN0IiBpbiBvYmo6DQogICAgICAgIG9iaiA9IG9ialsic3RhdGVfZGljdCJdDQogICAgZWxpZiBpc2luc3RhbmNlKG9iaiwgZGljdCkgYW5kICJt")
parts.append("b2RlbF9zdGF0ZV9kaWN0IiBpbiBvYmo6DQogICAgICAgIG9iaiA9IG9ialsibW9kZWxfc3RhdGVfZGljdCJdDQogICAgZWxpZiBpc2luc3RhbmNlKG9iaiwgZGljdCkgYW5kICJtb2RlbCIgaW4gb2JqIGFuZCBpc2luc3RhbmNlKG9ialsibW9kZWwiXSwgZGljdCk6")
parts.append("DQogICAgICAgIG9iaiA9IG9ialsibW9kZWwiXQ0KICAgIGlmIG5vdCBpc2luc3RhbmNlKG9iaiwgZGljdCk6DQogICAgICAgIHJldHVybiBOb25lDQogICAgIyDliaXluLjop4HliY3nvIA6bW9kdWxlLiAvIG1vZGVsLiAvIGJhY2tib25lLijoi6XmlbTkvZPluKbl")
parts.append("iY3nvIAs57uf5LiA6L+Y5Y6fKQ0KICAgIG91dCA9IHt9DQogICAgZm9yIGssIHYgaW4gb2JqLml0ZW1zKCk6DQogICAgICAgIGZvciBwcmUgaW4gKCJtb2R1bGUuIiwgIm1vZGVsLiIsICJiYWNrYm9uZS4iKToNCiAgICAgICAgICAgIGlmIGsuc3RhcnRzd2l0aChw")
parts.append("cmUpOg0KICAgICAgICAgICAgICAgIGsgPSBrW2xlbihwcmUpOl0NCiAgICAgICAgICAgICAgICBicmVhaw0KICAgICAgICBvdXRba10gPSB2DQogICAgcmV0dXJuIG91dA0KDQoNCmRlZiBfdHJ5X2xvYWRfb2ZmbGluZShuZXQsIHBhdGhzKToNCiAgICAiIiLmjInl")
parts.append("uo/lsJ3or5XmiorlgJnpgInnprvnur/mnYPph43liqDovb3ov5sgbmV0KHN0YXRlX2RpY3Qg57uT5p6E6aG75LiOIFJlc05ldDM0IOWMuemFjSnjgIINCg0KICAgIOmAgumFjeinhOWImSjpnaLlkJHnpL7ljLogUlNOQSDmnYPph40s5aaCIGJhcnVuMjEwNCDnmoQg")
parts.append("YmVzdF9yZXNuZXQzNF9mb2xkMC5wdGgpOg0KICAgIC0gY29udjEud2VpZ2h0IOmAmumBk+aVsOS4jeWQjOaXtjpyZXNoYXBlK21lYW4g6YCC6YWN5YiwIG5ldCDnmoQgY29udjEg6YCa6YGT5pWwKOWmgiAyNOKGkjMsDQogICAgICBiYXJ1bjIxMDQg5pivIDggY2xp")
parts.append("cCDDlyAzIOmAmumBk+eahCAyLjVEIOaooeWeiyk7DQogICAgLSBmYy4qIC8gaGVhZC4qIC8gY2xhc3NpZmllci4qIOS4gOW+i+WJpeemuyjmiJHku6znmoTmqKHlnovkvJrmm7/mjaLmiJAgMTIg5aS0KTsNCiAgICDov5Tlm54gKOaYr+WQpuaIkOWKnywg5ZG95Lit")
parts.append("5o+P6L+wKeOAgg0KICAgICIiIg0KICAgIGZvciBwLCBkZXNjIGluIHBhdGhzOg0KICAgICAgICBzZCA9IF9sb2FkX3N0YXRlX2RpY3RfZnJvbV9maWxlKHApDQogICAgICAgIGlmIHNkIGlzIE5vbmU6DQogICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICAjIDEp")
parts.append("IOmAgumFjSBjb252MTrpgJrpgZPmlbDkuI3lkIzliJkgcmVzaGFwZSttZWFuKOWmgiAyNCDpgJrpgZMg4oaSIDMg6YCa6YGTKQ0KICAgICAgICBpZiAiY29udjEud2VpZ2h0IiBpbiBzZDoNCiAgICAgICAgICAgIHcgPSBzZFsiY29udjEud2VpZ2h0Il0NCiAgICAg")
parts.append("ICAgICAgIHRhcmdldCA9IG5ldC5jb252MS53ZWlnaHQuc2hhcGVbMV0NCiAgICAgICAgICAgIGlmIHcuc2hhcGVbMV0gIT0gdGFyZ2V0IGFuZCB3LnNoYXBlWzFdICUgdGFyZ2V0ID09IDA6DQogICAgICAgICAgICAgICAgQiwgQywgSCwgVyA9IHcuc2hhcGUNCiAg")
parts.append("ICAgICAgICAgICAgICB3MiA9IHcucmVzaGFwZShCLCB0YXJnZXQsIEMgLy8gdGFyZ2V0LCBILCBXKS5tZWFuKGRpbT0yKQ0KICAgICAgICAgICAgICAgIHNkWyJjb252MS53ZWlnaHQiXSA9IHcyDQogICAgICAgICAgICAgICAgcHJpbnQoZiJb5p2D6YeNXSB7ZGVz")
parts.append("Y30gY29udjEg6YCa6YGTIHtDfeKGknt0YXJnZXR9IOW5s+Wdh+mAgumFjSIsIGZsdXNoPVRydWUpDQogICAgICAgICAgICBlbGlmIHcuc2hhcGVbMV0gIT0gdGFyZ2V0Og0KICAgICAgICAgICAgICAgIHNkLnBvcCgiY29udjEud2VpZ2h0IikgICMg5peg5rOV6YCC")
parts.append("6YWNLOS4ouW8gyjnlKjpmo/mnLrliJ3lp4vljJYpDQogICAgICAgICMgMikg5Yml56a7IGZjL2hlYWQvY2xhc3NpZmllcijmqKHlnovkvJrmm7/mjaLmiJAgMTIg5aS0KQ0KICAgICAgICBmb3IgayBpbiBsaXN0KHNkLmtleXMoKSk6DQogICAgICAgICAgICBpZiBr")
parts.append("LnN0YXJ0c3dpdGgoKCJmYy4iLCAiaGVhZC4iLCAiY2xhc3NpZmllci4iKSk6DQogICAgICAgICAgICAgICAgc2QucG9wKGspDQogICAgICAgIHRyeToNCiAgICAgICAgICAgIG1pc3NpbmcsIHVuZXhwZWN0ZWQgPSBuZXQubG9hZF9zdGF0ZV9kaWN0KHNkLCBzdHJp")
parts.append("Y3Q9RmFsc2UpDQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToNCiAgICAgICAgICAgIHByaW50KGYiW+adg+mHjV0ge2Rlc2N9IOS4jiBSZXNOZXQzNCDnu5PmnoTkuI3ljLnphY06IHtlfSIsIGZsdXNoPVRydWUpDQogICAgICAgICAgICBjb250aW51ZQ0K")
parts.append("ICAgICAgICBleGVtcHQgPSB7ayBmb3IgayBpbiBtaXNzaW5nIGlmIGsuc3RhcnRzd2l0aCgoImNvbnYxLiIsICJmYy4iKSl9DQogICAgICAgIGlmIG5vdCB1bmV4cGVjdGVkIGFuZCBsZW4obWlzc2luZykgPT0gbGVuKGV4ZW1wdCk6DQogICAgICAgICAgICBwcmlu")
parts.append("dChmIlvmnYPph41dIOemu+e6v+mihOiuree7g+WKoOi9veaIkOWKnzoge2Rlc2N9ICjopobnm5Yge2xlbihzZCl9IOS4quWPguaVsOW8oOmHjykiLCBmbHVzaD1UcnVlKQ0KICAgICAgICAgICAgcmV0dXJuIFRydWUsIGRlc2MNCiAgICAgICAgcHJpbnQoZiJb5p2D")
parts.append("6YeNXSB7ZGVzY30g57uT5p6E5LiN5Yy56YWNKG1pc3Npbmcge2xlbihtaXNzaW5nKSAtIGxlbihleGVtcHQpfSDkuKrpnZ7osYHlhY0ga2V5LCAiDQogICAgICAgICAgICAgIGYidW5leHBlY3RlZCB7bGVuKHVuZXhwZWN0ZWQpfSks5bCd6K+V5LiL5LiA5LiqIiwg")
parts.append("Zmx1c2g9VHJ1ZSkNCiAgICByZXR1cm4gRmFsc2UsICIiDQoNCg0KZGVmIF9zY2FuX3BzZXVkbyhiYXNlLCBkZXB0aD0wLCBtYXhfZGVwdGg9Myk6DQogICAgIiIi6YCS5b2S5Zyo5oyC6L2955uu5b2V5LiL5om+6L2v5qCH562+IENTVijpmZDliLbmt7HluqYs6Lez")
parts.append("6L+HIERJQ09NIOebruW9lSnjgIINCg0KICAgIOi/lOWbniAoRGF0YUZyYW1lIOaIliBOb25lLCDmnaXmupDmj4/ov7Ap44CC6K+G5Yir6KeE5YiZOuWQqyBTdHVkeUluc3RhbmNlVUlEIOS4lOiHs+WwkeacieS4gOS4qg0KICAgIHBzZXVkb18g5YmN57yA5YiXKOWm")
parts.append("giBwc2V1ZG9fQUNMKeOAguaJvuS4jeWIsOi/lOWbniAoTm9uZSwgIiIp44CCDQoNCiAgICDor4bliKvmlrnlvI/lj4zpgJrpgZM6DQogICAgMS4g5paH5Lu25ZCN55m95ZCN5Y2VKF9QU0VVRE9fRklMRVMp5b+r6YCf5ZG95LitOw0KICAgIDIuICoq5oyJ5YaF5a65")
parts.append("6K+G5YirKio65Lu75oSPICouY3N2IOWPquimgeihqOWktOWQqyBTdHVkeUluc3RhbmNlVUlEICsgcHNldWRvXyog5YiX5Y2z5ZG95LitDQogICAgICAgKOS4jeS+nei1luaWh+S7tuWQjSznpL7ljLrmlbDmja7pm4bmlLnlkI0v5o2i57uT5p6E5Lmf6IO95omr5Yiw")
parts.append("KeOAgg0KICAgICIiIg0KICAgIGlmIGRlcHRoID4gbWF4X2RlcHRoIG9yIG5vdCBvcy5wYXRoLmlzZGlyKGJhc2UpOg0KICAgICAgICByZXR1cm4gTm9uZSwgIiINCiAgICB0cnk6DQogICAgICAgIGVudHJpZXMgPSBzb3J0ZWQob3MubGlzdGRpcihiYXNlKSkNCiAg")
parts.append("ICBleGNlcHQgT1NFcnJvcjoNCiAgICAgICAgcmV0dXJuIE5vbmUsICIiDQogICAgIyDpgJrpgZMgMTrmlofku7blkI3nmb3lkI3ljZUo5ZCR5ZCO5YW85a65LOivu+WFqOmHjykNCiAgICBmb3IgZm4gaW4gX1BTRVVET19GSUxFUzoNCiAgICAgICAgcCA9IG9zLnBh")
parts.append("dGguam9pbihiYXNlLCBmbikNCiAgICAgICAgaWYgb3MucGF0aC5leGlzdHMocCkgYW5kIG9zLnBhdGguaXNmaWxlKHApOg0KICAgICAgICAgICAgdHJ5Og0KICAgICAgICAgICAgICAgIGNhbmQgPSBwZC5yZWFkX2NzdihwKQ0KICAgICAgICAgICAgZXhjZXB0IEV4")
parts.append("Y2VwdGlvbjoNCiAgICAgICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICAgICAgaWYgIlN0dWR5SW5zdGFuY2VVSUQiIGluIGNhbmQuY29sdW1ucyBhbmQgYW55KA0KICAgICAgICAgICAgICAgICAgICBjLnN0YXJ0c3dpdGgoInBzZXVkb18iKSBmb3IgYyBpbiBj")
parts.append("YW5kLmNvbHVtbnMpOg0KICAgICAgICAgICAgICAgIHJldHVybiBjYW5kLCBvcy5wYXRoLnJlbHBhdGgocCwgIi9rYWdnbGUvaW5wdXQiKQ0KICAgICMg6YCa6YGTIDI65oyJ5YaF5a656K+G5YirKOWPquivu+ihqOWktCBucm93cz0zIOWIpOaWrSzlkb3kuK3lho3l")
parts.append("hajor7spDQogICAgZm9yIGZuIGluIGVudHJpZXM6DQogICAgICAgIGlmIG5vdCBmbi5sb3dlcigpLmVuZHN3aXRoKCIuY3N2Iik6DQogICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICBwID0gb3MucGF0aC5qb2luKGJhc2UsIGZuKQ0KICAgICAgICBpZiBub3Qg")
parts.append("b3MucGF0aC5pc2ZpbGUocCk6DQogICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICB0cnk6DQogICAgICAgICAgICBoZWFkID0gcGQucmVhZF9jc3YocCwgbnJvd3M9MykNCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoNCiAgICAgICAgICAgIGNvbnRpbnVlDQog")
parts.append("ICAgICAgIGlmICJTdHVkeUluc3RhbmNlVUlEIiBpbiBoZWFkLmNvbHVtbnMgYW5kIGFueSgNCiAgICAgICAgICAgICAgICBjLnN0YXJ0c3dpdGgoInBzZXVkb18iKSBmb3IgYyBpbiBoZWFkLmNvbHVtbnMpOg0KICAgICAgICAgICAgdHJ5Og0KICAgICAgICAgICAg")
parts.append("ICAgIGNhbmQgPSBwZC5yZWFkX2NzdihwKQ0KICAgICAgICAgICAgICAgIHJldHVybiBjYW5kLCBvcy5wYXRoLnJlbHBhdGgocCwgIi9rYWdnbGUvaW5wdXQiKQ0KICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoNCiAgICAgICAgICAgICAgICBjb250aW51ZQ0K")
parts.append("ICAgIGZvciBzdWIgaW4gZW50cmllczoNCiAgICAgICAgaWYgc3ViIGluIF9TS0lQX0RJUlM6DQogICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICBjYW5kLCBzcmMgPSBfc2Nhbl9wc2V1ZG8ob3MucGF0aC5qb2luKGJhc2UsIHN1YiksIGRlcHRoICsgMSwgbWF4")
parts.append("X2RlcHRoKQ0KICAgICAgICBpZiBjYW5kIGlzIG5vdCBOb25lOg0KICAgICAgICAgICAgcmV0dXJuIGNhbmQsIHNyYw0KICAgIHJldHVybiBOb25lLCAiIg0KDQoNCmRlZiBmaW5kX3BzZXVkb19kYXRhc2V0KCk6DQogICAgIiIi5ZyoIC9rYWdnbGUvaW5wdXQg5LiL")
parts.append("6YCS5b2S5omr5o+P56S+5Yy66L2v5qCH562+5pWw5o2u6ZuGKExMTSDku47miqXlkYrmj5Dlj5bnmoTkvKrmoIfnrb4p44CCDQoNCiAgICDlhbzlrrnljZXlsYIoL2thZ2dsZS9pbnB1dC88c2x1Zz4vKeS4juWkmuWxgigva2FnZ2xlL2lucHV0L2RhdGFzZXRzLzxz")
parts.append("bHVnPi8p5oyC6L2944CCDQogICAg6L+U5ZueIChEYXRhRnJhbWUg5oiWIE5vbmUsIOadpea6kOaPj+i/sCk75om+5LiN5Yiw6L+U5ZueIChOb25lLCAiIinjgIINCiAgICAiIiINCiAgICBpZiBub3Qgb3MucGF0aC5pc2RpcigiL2thZ2dsZS9pbnB1dCIpOg0KICAg")
parts.append("ICAgICByZXR1cm4gTm9uZSwgIiINCiAgICByZXR1cm4gX3NjYW5fcHNldWRvKCIva2FnZ2xlL2lucHV0IikNCg0KDQpkZWYgbG9hZF9sYWJlbHMocm9vdCwgcHNldWRvX2NvbmY9MC4wLCB1c2VfcmVwb3J0X2xhYmVscz1UcnVlKToNCiAgICAiIiLov5Tlm54gdHJh")
parts.append("aW4gc3R1ZHkg57qn5qCH562+IERhdGFGcmFtZSjooYw9U3R1ZHlJbnN0YW5jZVVJRCwg5YiXPUxBQkVMUywg5YiG5pWwL05hTinjgIINCg0KICAgIOagh+etvuadpea6kOS8mOWFiOe6pygyMDI2LTA5LTA2IOi1tzvml6fnmoQi56S+5Yy66L2v5qCH562+5pWw5o2u")
parts.append("6ZuGIuaWueahiOW3suivgeS8quW8g+eUqOKAlOKAlA0KICAgIGJhcnVuMjEwNCDnmoQgcHNldWRvXyogOTguNyUg5piv5a2X6Z2iIDAuNSDmrbvlgLzjgIHpm7blop7ph4/kv6Hmga8s5pivIGxvc3Mg5oGSIDAuNjkzDQogICAg55qE55yf5qC55ZugKToNCiAgICAx")
parts.append("LiDlrpjmlrkgdHJhaW4uY3N2IOagh+azqCgwLzEs5LuFIDU4LzQ0MDcg5L6LKTsNCiAgICAyLiDlhoXnva7miqXlkYrop4TliJnmir3lj5blmaggcnNuYV9yZXBvcnRfbGFiZWxzKOenu+akjeiHqiByc25hLWtuZWUtYmFzZWxpbmUtdjEsDQogICAgICAg5aSa6K+t")
parts.append("6KiAL+aMieWPpeWQpuWumuaegeaApy/kuKXph43luqbliIbnuqcpOuWvueavj+S7veaUvuWwhOaKpeWRiuS6p+WHuiAxMiDnl4XngbYgc29mdCDliIbmlbANCiAgICAgICAo6Ziz5oCnIDAuNzJ+MC45NSAvIOmYtOaApyAwLjA0fjAuMjAgLyDml6Dor4Hmja49TmFO")
parts.append("KTsNCiAgICAgICDlrpjmlrnnvLrlpLHmoLznlKjmir3lj5bliIbmlbDloavlhYUs5peg6K+B5o2u5qC85L+d5oyBIE5hTijkuI3ov5vmjZ/lpLEs55SxIHZhbGlkIOaOqeeggeaOkumZpCnjgIINCiAgICAiIiINCiAgICB0cmFpbiA9IHBkLnJlYWRfY3N2KG9zLnBh")
parts.append("dGguam9pbihyb290LCAidHJhaW4uY3N2IikpDQogICAgZGYgPSB0cmFpbltbIlN0dWR5SW5zdGFuY2VVSUQiXSArIExBQkVMU10uY29weSgpDQogICAgb2ZmaWNpYWxfY250ID0ge2M6IGludChkZltjXS5ub3RuYSgpLnN1bSgpKSBmb3IgYyBpbiBMQUJFTFN9DQoN")
parts.append("CiAgICAjIEVYUC0wMTI6IExMTS1wcmltYXJ5IHNvZnQgbGFiZWxzIChnb2xkIHdpbnM7IHNpbGVuY2UgMC4yNS8wLjUg4oaSIE5hTikNCiAgICBleHAwMTIgPSBvcy5lbnZpcm9uLmdldCgiRVhQMDEyX0xMTV9QUklNQVJZIiwgIiIpDQogICAgaWYgZXhwMDEyOg0K")
parts.append("ICAgICAgICBmcm9tIGV4cDAxMl9sbG1fcHJpbWFyeSBpbXBvcnQgYnVpbGRfZXhwMDEyX2xhYmVscw0KICAgICAgICBvdXQgPSBidWlsZF9leHAwMTJfbGFiZWxzKG9zLnBhdGguam9pbihyb290LCAidHJhaW4uY3N2IiksIHZlcmJvc2U9VHJ1ZSkNCiAgICAgICAg")
parts.append("cHJpbnQoIlvmoIfnrb5dIEVYUC0wMTIgTExNLXByaW1hcnkgdGFibGUgbG9hZGVkIikNCiAgICAgICAgcmV0dXJuIG91dA0KDQogICAgaWYgbm90IHVzZV9yZXBvcnRfbGFiZWxzOg0KICAgICAgICBwcmludCgiW+agh+etvl0gLS1uby1yZXBvcnQtbGFiZWxzOuS7")
parts.append("heWumOaWueagh+azqOWPr+eUqCIpDQogICAgICAgIHJldHVybiBkZg0KICAgIGlmIG5vdCBIQVNfUkw6DQogICAgICAgIHByaW50KCJb5qCH562+XSDmnKrmib7liLAgcnNuYV9yZXBvcnRfbGFiZWxzLnB5KOmcgOS4jiByc25hX3YzLnB5IOWQjOebruW9lSksIg0K")
parts.append("ICAgICAgICAgICAgICAi5LuF5a6Y5pa55qCH5rOo5Y+v55SoIikNCiAgICAgICAgcmV0dXJuIGRmDQogICAgaWYgIlJlcG9ydCIgbm90IGluIHRyYWluLmNvbHVtbnM6DQogICAgICAgIHByaW50KCJb5qCH562+XSB0cmFpbi5jc3Yg5pegIFJlcG9ydCDliJcs5LuF")
parts.append("5a6Y5pa55qCH5rOo5Y+v55SoIikNCiAgICAgICAgcmV0dXJuIGRmDQoNCiAgICBwc2V1ZG8gPSBSTC5idWlsZF9wc2V1ZG9fdGFibGUodHJhaW4sIHZlcmJvc2U9VHJ1ZSkNCiAgICBwc2V1ZG8gPSBwc2V1ZG8uc2V0X2luZGV4KCJTdHVkeUluc3RhbmNlVUlEIikN")
parts.append("CiAgICBwcmludChmIlvmoIfnrb5dIOWhq+WFhee7n+iuoSh7bGVuKGRmKX0g5LiqIHN0dWR5KToiKQ0KICAgIGZvciBjIGluIExBQkVMUzoNCiAgICAgICAgbWFzayA9IGRmW2NdLmlzbmEoKS52YWx1ZXMNCiAgICAgICAgaWYgbm90IG1hc2suYW55KCk6DQogICAg")
parts.append("ICAgICAgICBwcmludChmIiAge2M6MThzfSDlrpjmlrnlhajph48s5peg6ZyA5aGr5YWFIikNCiAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgIHVpZHMgPSBkZi5sb2NbbWFzaywgIlN0dWR5SW5zdGFuY2VVSUQiXQ0KICAgICAgICB2YWxzID0gcHNldWRvWyJw")
parts.append("c2V1ZG9fIiArIGNdLnJlaW5kZXgodWlkcykudmFsdWVzDQogICAgICAgIG5fZmlsbCA9IGludChwZC5ub3RuYSh2YWxzKS5zdW0oKSkNCiAgICAgICAgZGYubG9jW21hc2ssIGNdID0gdmFscw0KICAgICAgICBwcmludChmIiAge2M6MThzfSDlrpjmlrkge29mZmlj")
parts.append("aWFsX2NudFtjXTo0ZH0gKyDmiqXlkYrmir3lj5bloavlhYUge25fZmlsbDo0ZH0iDQogICAgICAgICAgICAgIGYiID0g5pyJ5pWIIHtpbnQoZGZbY10ubm90bmEoKS5zdW0oKSk6NWR9IikNCiAgICByZXR1cm4gZGYNCg0KDQpfRENNX01JU1NfUFJJTlRFRCA9IFtG")
parts.append("YWxzZV0gICMg5Y+q5omT5Y2w5LiA5qyhIERJQ09NIOacquWRveS4reiviuaWrSzpgb/lhY0gMiDkuIcgc2VyaWVzIOWIt+Wxjw0KDQoNCmRlZiByZWFkX2RpY29tX3NsaWNlcyhkY21fZGlyLCBzZXJpZXNfdWlkLCBzdHVkeV91aWQ9Tm9uZSk6DQogICAgIiIi6K+7")
parts.append("5Y+W5LiA5LiqIHNlcmllcyDnmoTmiYDmnInliIfniYcs5oyJ54mp55CG5L2N572u5o6S5bqPLOi/lOWbniBbKHBhdGgsIHopXeOAgg0KDQogICAg6Lev5b6E5YW85a655aSa56eNIEthZ2dsZSDluIPlsYAo5a6e5rWLIFJTTkEgMjAyNiDkuLrkuKTlsYIpOg0KICAg")
parts.append("ICAgMSkgPGRjbV9kaXI+LzxzZXJpZXNfdWlkPi8gICAgICAgICAgICAgICAgKOaXp+WBh+iuvjpzZXJpZXMg55uu5b2V55u05o6l5oyCKQ0KICAgICAgMikgPGRjbV9kaXI+LzxzdHVkeV91aWQ+LzxzZXJpZXNfdWlkPi8gICAgKFJTTkEg5qCH5YeGOnN0dWR5IOeb")
parts.append("ruW9leS4i+WGjeaMgiBzZXJpZXMpDQogICAg5om+5LiN5Yiw5oiW6K+75LiN5Yiw6L+U5ZueIFtd44CCDQogICAgIiIiDQogICAgY2FuZHMgPSBbb3MucGF0aC5qb2luKGRjbV9kaXIsIHNlcmllc191aWQpXQ0KICAgIGlmIHN0dWR5X3VpZDoNCiAgICAgICAgY2Fu")
parts.append("ZHMuYXBwZW5kKG9zLnBhdGguam9pbihkY21fZGlyLCBzdHVkeV91aWQsIHNlcmllc191aWQpKQ0KICAgIHNkaXIgPSBuZXh0KChjIGZvciBjIGluIGNhbmRzIGlmIG9zLnBhdGguaXNkaXIoYykpLCBOb25lKQ0KICAgIGlmIHNkaXIgaXMgTm9uZToNCiAgICAgICAg")
parts.append("aWYgbm90IF9EQ01fTUlTU19QUklOVEVEWzBdOg0KICAgICAgICAgICAgX0RDTV9NSVNTX1BSSU5URURbMF0gPSBUcnVlDQogICAgICAgICAgICBwcmludChmIlvor4rmlq0tRElDT01dIOaJvuS4jeWIsCBzZXJpZXMg55uu5b2VLOWwneivlei/hzoge2NhbmRzfSIs")
parts.append("IGZsdXNoPVRydWUpDQogICAgICAgICAgICBwcmludChmIlvor4rmlq0tRElDT01dIHtkY21fZGlyfSDmmK/lkKblrZjlnKg6IHtvcy5wYXRoLmlzZGlyKGRjbV9kaXIpfSIsIGZsdXNoPVRydWUpDQogICAgICAgICAgICB0cnk6DQogICAgICAgICAgICAgICAgaW5u")
parts.append("ZXIgPSBzb3J0ZWQob3MubGlzdGRpcihkY21fZGlyKSlbOjhdIGlmIG9zLnBhdGguaXNkaXIoZGNtX2RpcikgZWxzZSBbXQ0KICAgICAgICAgICAgICAgIHByaW50KGYiW+iviuaWrS1ESUNPTV0ge2RjbV9kaXJ9IOWunumZheWGheWuueagt+acrDoge2lubmVyfSIs")
parts.append("IGZsdXNoPVRydWUpDQogICAgICAgICAgICBleGNlcHQgT1NFcnJvcjoNCiAgICAgICAgICAgICAgICBwYXNzDQogICAgICAgIHJldHVybiBbXQ0KICAgIGZpbGVzID0gW29zLnBhdGguam9pbihzZGlyLCBmKSBmb3IgZiBpbiBvcy5saXN0ZGlyKHNkaXIpIGlmIGYu")
parts.append("bG93ZXIoKS5lbmRzd2l0aCgiLmRjbSIpXQ0KICAgIGlmIG5vdCBmaWxlczoNCiAgICAgICAgaWYgbm90IF9EQ01fTUlTU19QUklOVEVEWzBdOg0KICAgICAgICAgICAgX0RDTV9NSVNTX1BSSU5URURbMF0gPSBUcnVlDQogICAgICAgICAgICBwcmludChmIlvor4rm")
parts.append("lq0tRElDT01dIOebruW9leWtmOWcqOS9huaXoCAuZGNtIOaWh+S7tjoge3NkaXJ9IiwgZmx1c2g9VHJ1ZSkNCiAgICAgICAgICAgIHByaW50KGYiW+iviuaWrS1ESUNPTV0g55uu5b2V5YaF5a655qC35pysOiB7c29ydGVkKG9zLmxpc3RkaXIoc2RpcikpWzo4XX0i")
parts.append("LCBmbHVzaD1UcnVlKQ0KICAgICAgICByZXR1cm4gW10NCiAgICByb3dzID0gW10NCiAgICBmb3IgZiBpbiBmaWxlczoNCiAgICAgICAgdHJ5Og0KICAgICAgICAgICAgZCA9IHB5ZGljb20uZGNtcmVhZChmLCBzdG9wX2JlZm9yZV9waXhlbHM9RmFsc2UpDQogICAg")
parts.append("ICAgICAgICBwb3MgPSBkLmdldCgiSW1hZ2VQb3NpdGlvblBhdGllbnQiKQ0KICAgICAgICAgICAgeiA9IGZsb2F0KHBvc1syXSkgaWYgcG9zIGlzIG5vdCBOb25lIGFuZCBsZW4ocG9zKSA+PSAzIGVsc2UgTm9uZQ0KICAgICAgICAgICAgcm93cy5hcHBlbmQoKGYs")
parts.append("IHopKQ0KICAgICAgICBleGNlcHQgRXhjZXB0aW9uOg0KICAgICAgICAgICAgY29udGludWUNCiAgICByb3dzLnNvcnQoa2V5PWxhbWJkYSByOiAoclsxXSBpcyBOb25lLCByWzFdIGlmIHJbMV0gaXMgbm90IE5vbmUgZWxzZSAwLjApKQ0KICAgIHJldHVybiByb3dz")
parts.append("DQoNCg0KIyAtLS0tLS0tLS0tLS0tLS0tIERJQ09NIC0+IDIuNUQgY2xpcCDpooTlpITnkIYgLS0tLS0tLS0tLS0tLS0tLQ0KDQpkZWYgd2luZG93X2FuZF9yZXNpemUoZGNtLCBpbWdfc2l6ZT1JTUdfU0laRSk6DQogICAgIiIi5Y2V5byg5YiH54mHOlJlc2NhbGUg")
parts.append("5ZCO55m+5YiG5L2N56qX5Y+j5YyWIC0+IFswLDFdIC0+IHJlc2l6ZSAyMjTjgIIiIiINCiAgICBhcnIgPSBkY20ucGl4ZWxfYXJyYXkuYXN0eXBlKG5wLmZsb2F0MzIpDQogICAgc2xvcGUgPSBmbG9hdChkY20uZ2V0KCJSZXNjYWxlU2xvcGUiLCAxKSkNCiAgICBp")
parts.append("bnRlciA9IGZsb2F0KGRjbS5nZXQoIlJlc2NhbGVJbnRlcmNlcHQiLCAwKSkNCiAgICBhcnIgPSBhcnIgKiBzbG9wZSArIGludGVyDQogICAgaWYgZGNtLmdldCgiUGhvdG9tZXRyaWNJbnRlcnByZXRhdGlvbiIpID09ICJNT05PQ0hST01FMSI6DQogICAgICAgIGFy")
parts.append("ciA9IGFyci5tYXgoKSAtIGFycg0KICAgIGxvLCBoaSA9IG5wLnBlcmNlbnRpbGUoYXJyLCBbMSwgOTldKQ0KICAgIGFyciA9IG5wLmNsaXAoKGFyciAtIGxvKSAvIG1heChoaSAtIGxvLCAxZS02KSwgMCwgMSkNCiAgICBmcm9tIFBJTCBpbXBvcnQgSW1hZ2UNCiAg")
parts.append("ICBpbWcgPSBJbWFnZS5mcm9tYXJyYXkoKGFyciAqIDI1NSkuYXN0eXBlKG5wLnVpbnQ4KSkucmVzaXplKChpbWdfc2l6ZSwgaW1nX3NpemUpLCBJbWFnZS5CSUxJTkVBUikNCiAgICByZXR1cm4gbnAuYXNhcnJheShpbWcsIGR0eXBlPW5wLmZsb2F0MzIpIC8gMjU1")
parts.append("LjANCg0KDQpkZWYgbWFrZV9jbGlwcyhzZXJpZXNfcm93cywgbl9jbGlwcz0yLCBpbWdfc2l6ZT1JTUdfU0laRSk6DQogICAgIiIi5LuO5o6S5bqP5aW955qE5YiH54mH5YiX6KGo55Sf5oiQ6Iul5bmyIDMg6YCa6YGTIGNsaXAo55u46YK75LiJ5bygKeOAgiIiIg0K")
parts.append("ICAgIGNsaXBzID0gW10NCiAgICBwYXRocyA9IFtyWzBdIGZvciByIGluIHNlcmllc19yb3dzXQ0KICAgIGlmIGxlbihwYXRocykgPCBDTElQOg0KICAgICAgICByZXR1cm4gY2xpcHMNCiAgICBzdGVwID0gbWF4KDEsIChsZW4ocGF0aHMpIC0gMikgLy8gbl9jbGlw")
parts.append("cykNCiAgICBmb3IgYyBpbiByYW5nZSgxLCBsZW4ocGF0aHMpIC0gMSwgc3RlcCk6DQogICAgICAgIGlmIGxlbihjbGlwcykgPj0gbl9jbGlwczoNCiAgICAgICAgICAgIGJyZWFrDQogICAgICAgIGNoYW4gPSBbXQ0KICAgICAgICBvayA9IFRydWUNCiAgICAgICAg")
parts.append("Zm9yIGsgaW4gKGMgLSAxLCBjLCBjICsgMSk6DQogICAgICAgICAgICB0cnk6DQogICAgICAgICAgICAgICAgZGNtID0gcHlkaWNvbS5kY21yZWFkKHBhdGhzW2tdKQ0KICAgICAgICAgICAgICAgIGNoYW4uYXBwZW5kKHdpbmRvd19hbmRfcmVzaXplKGRjbSwgaW1n")
parts.append("X3NpemUpKQ0KICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoNCiAgICAgICAgICAgICAgICBvayA9IEZhbHNlDQogICAgICAgICAgICAgICAgYnJlYWsNCiAgICAgICAgaWYgb2s6DQogICAgICAgICAgICBjbGlwcy5hcHBlbmQobnAuc3RhY2soY2hhbiwgYXhp")
parts.append("cz0wKSkgICMgKDMsIEgsIFcpDQogICAgcmV0dXJuIGNsaXBzDQoNCg0KaWYgSEFTX1RPUkNIOg0KICAgICMgLS0tLS0tLS0tLS0tLS0tLSDmlbDmja7pm4YodjI65oyJ5bmz6Z2i6L+H5rukIHNlcmllcykgLS0tLS0tLS0tLS0tLS0tLQ0KDQogICAgY2xhc3MgS25l")
parts.append("ZURhdGFzZXQoRGF0YXNldCk6DQogICAgICAgICIiInN0dWR5IOe6p+aVsOaNrumbhizlj6rlj5bmjIflrpogQW5hdG9taWNhbF9QbGFuZSDnmoQgc2VyaWVzLOaLvOaIkCAoQyozLCBILCBXKeOAgiIiIg0KDQogICAgICAgIGRlZiBfX2luaXRfXyhzZWxmLCBzdHVk")
parts.append("aWVzLCBsYWJlbHNfZGYsIGRjbV9yb290LCBzZXJpZXNfZGYsIHBsYW5lLA0KICAgICAgICAgICAgICAgICAgICAgaGFzX3RhcmdldHM9VHJ1ZSwgbl9jbGlwcz0yKToNCiAgICAgICAgICAgIHNlbGYuc3R1ZGllcyA9IHN0dWRpZXMNCiAgICAgICAgICAgIHNlbGYu")
parts.append("bGFiZWxzX2RmID0gbGFiZWxzX2RmDQogICAgICAgICAgICBzZWxmLmRjbV9yb290ID0gZGNtX3Jvb3QNCiAgICAgICAgICAgIHNlbGYuc2VyaWVzX2RmID0gc2VyaWVzX2RmDQogICAgICAgICAgICBzZWxmLnBsYW5lID0gcGxhbmUNCiAgICAgICAgICAgIHNlbGYu")
parts.append("aGFzX3RhcmdldHMgPSBoYXNfdGFyZ2V0cw0KICAgICAgICAgICAgc2VsZi5uX2NsaXBzID0gbl9jbGlwcw0KICAgICAgICAgICAgc2VsZi5jYWNoZSA9IHt9ICAjIOavj+aKmOeLrOeri+e8k+WtmDpmb2xkIOe7k+adn+majyBkYXRhc2V0IOmHiuaUvizpgb/lhY0g")
parts.append("NSDmipjntK/np68gT09NDQogICAgICAgICAgICAjIOiviuaWreiuoeaVsDror7vliLAgY2xpcCDnmoQgc3R1ZHkgLyDnqbogc3R1ZHkgLyDntK/orqEgY2xpcCDmlbANCiAgICAgICAgICAgIHNlbGYubl9oaXQgPSAwDQogICAgICAgICAgICBzZWxmLm5fZW1wdHkg")
parts.append("PSAwDQogICAgICAgICAgICBzZWxmLm5fY2xpcHNfdG90YWwgPSAwDQoNCiAgICAgICAgZGVmIHN0YXRzKHNlbGYpOg0KICAgICAgICAgICAgIiIi6L+U5ZueICjor7vliLAgY2xpcCDnmoQgc3R1ZHkg5pWwLCDnqbogc3R1ZHkg5pWwLCDlubPlnYcgY2xpcCDmlbAp")
parts.append("LOmYsuatoumdmem7mOeZveiuree7g+OAgiIiIg0KICAgICAgICAgICAgdG90YWwgPSBzZWxmLm5faGl0ICsgc2VsZi5uX2VtcHR5DQogICAgICAgICAgICBhdmcgPSAoc2VsZi5uX2NsaXBzX3RvdGFsIC8gc2VsZi5uX2hpdCkgaWYgc2VsZi5uX2hpdCBlbHNlIDAu")
parts.append("MA0KICAgICAgICAgICAgcmV0dXJuIHNlbGYubl9oaXQsIHNlbGYubl9lbXB0eSwgYXZnDQoNCiAgICAgICAgZGVmIF9sb2FkX3N0dWR5KHNlbGYsIHVpZCk6DQogICAgICAgICAgICBrZXkgPSAodWlkLCBzZWxmLnBsYW5lKQ0KICAgICAgICAgICAgaWYga2V5IGlu")
parts.append("IHNlbGYuY2FjaGU6DQogICAgICAgICAgICAgICAgcmV0dXJuIHNlbGYuY2FjaGVba2V5XQ0KICAgICAgICAgICAgc3Jvd3MgPSBzZWxmLnNlcmllc19kZltzZWxmLnNlcmllc19kZlsiU3R1ZHlJbnN0YW5jZVVJRCJdID09IHVpZF0NCiAgICAgICAgICAgIGlmICJB")
parts.append("bmF0b21pY2FsX1BsYW5lIiBpbiBzcm93cy5jb2x1bW5zOg0KICAgICAgICAgICAgICAgIHNyb3dzID0gc3Jvd3Nbc3Jvd3NbIkFuYXRvbWljYWxfUGxhbmUiXSA9PSBzZWxmLnBsYW5lXQ0KICAgICAgICAgICAgICAgIHNyb3dzID0gc3Jvd3Muc29ydF92YWx1ZXMo")
parts.append("IkZsdWlkX1NlbnNpdGl2ZSIsIGtpbmQ9InN0YWJsZSIpLmRyb3BfZHVwbGljYXRlcygNCiAgICAgICAgICAgICAgICAgICAgc3Vic2V0PVsiRmx1aWRfU2Vuc2l0aXZlIl0pLmhlYWQoTUFYX1NFUklFUykNCiAgICAgICAgICAgIGVsc2U6DQogICAgICAgICAgICAg")
parts.append("ICAgc3Jvd3MgPSBzcm93cy5oZWFkKE1BWF9TRVJJRVMpDQogICAgICAgICAgICBjbGlwcyA9IFtdDQogICAgICAgICAgICBmb3IgXywgc3IgaW4gc3Jvd3MuaXRlcnJvd3MoKToNCiAgICAgICAgICAgICAgICBzaWQgPSBzclsiU2VyaWVzSW5zdGFuY2VVSUQiXQ0K")
parts.append("ICAgICAgICAgICAgICAgIHJvd3MgPSByZWFkX2RpY29tX3NsaWNlcyhzZWxmLmRjbV9yb290LCBzaWQsIHN0dWR5X3VpZD11aWQpDQogICAgICAgICAgICAgICAgY2xpcHMuZXh0ZW5kKG1ha2VfY2xpcHMocm93cywgbl9jbGlwcz1zZWxmLm5fY2xpcHMpKQ0KICAg")
parts.append("ICAgICAgICAgc2VsZi5jYWNoZVt1aWRdID0gY2xpcHMNCiAgICAgICAgICAgICMg5LuF6aaW5qyh5Yqg6L296K6h5pWwKOe8k+WtmOWRveS4reS4jeWGjee0r+WKoCzpgb/lhY3lpJogZXBvY2gg6YeN5aSNKQ0KICAgICAgICAgICAgaWYgY2xpcHM6DQogICAgICAg")
parts.append("ICAgICAgICAgc2VsZi5uX2hpdCArPSAxDQogICAgICAgICAgICAgICAgc2VsZi5uX2NsaXBzX3RvdGFsICs9IGxlbihjbGlwcykNCiAgICAgICAgICAgIGVsc2U6DQogICAgICAgICAgICAgICAgc2VsZi5uX2VtcHR5ICs9IDENCiAgICAgICAgICAgIHJldHVybiBj")
parts.append("bGlwcw0KDQogICAgICAgIGRlZiBfX2xlbl9fKHNlbGYpOg0KICAgICAgICAgICAgcmV0dXJuIGxlbihzZWxmLnN0dWRpZXMpDQoNCiAgICAgICAgZGVmIF9fZ2V0aXRlbV9fKHNlbGYsIGkpOg0KICAgICAgICAgICAgdWlkID0gc2VsZi5zdHVkaWVzW2ldDQogICAg")
parts.append("ICAgICAgICBjbGlwcyA9IHNlbGYuX2xvYWRfc3R1ZHkodWlkKSAgIyDlkb3kuK0v56m66K6h5pWw5ZyoIF9sb2FkX3N0dWR5IOmmluasoeWKoOi9veaXtuWujOaIkA0KICAgICAgICAgICAgaW5fY2ggPSBzZWxmLm5fY2xpcHMgKiBDTElQICAjIOi+k+WFpemAmumB")
parts.append("k+aVsDpjbGlwIOaVsCDDlyDmr48gY2xpcCAzIOWIh+eJhw0KICAgICAgICAgICAgaWYgbm90IGNsaXBzOg0KICAgICAgICAgICAgICAgIHggPSBucC56ZXJvcygoaW5fY2gsIElNR19TSVpFLCBJTUdfU0laRSksIGR0eXBlPW5wLmZsb2F0MzIpDQogICAgICAgICAg")
parts.append("ICBlbHNlOg0KICAgICAgICAgICAgICAgIHggPSBucC5jb25jYXRlbmF0ZShjbGlwc1s6IHNlbGYubl9jbGlwc10sIGF4aXM9MCkgICMgKGsqMywgSCwgVykNCiAgICAgICAgICAgICAgICBwYWQgPSBpbl9jaCAtIHguc2hhcGVbMF0NCiAgICAgICAgICAgICAgICBp")
parts.append("ZiBwYWQgPiAwOg0KICAgICAgICAgICAgICAgICAgICB4ID0gbnAuY29uY2F0ZW5hdGUoW3gsIG5wLnplcm9zKChwYWQsIElNR19TSVpFLCBJTUdfU0laRSksIGR0eXBlPW5wLmZsb2F0MzIpXSkNCiAgICAgICAgICAgICAgICBlbGlmIHBhZCA8IDA6DQogICAgICAg")
parts.append("ICAgICAgICAgICAgIHggPSB4WzogaW5fY2hdDQogICAgICAgICAgICB4ID0gdG9yY2gudGVuc29yKHgsIGR0eXBlPXRvcmNoLmZsb2F0MzIpDQogICAgICAgICAgICB5ID0gdG9yY2guemVyb3MobGVuKExBQkVMUyksIGR0eXBlPXRvcmNoLmZsb2F0MzIpDQogICAg")
parts.append("ICAgICAgICB2YWxpZCA9IHRvcmNoLnplcm9zKGxlbihMQUJFTFMpLCBkdHlwZT10b3JjaC5mbG9hdDMyKQ0KICAgICAgICAgICAgaWYgc2VsZi5oYXNfdGFyZ2V0czoNCiAgICAgICAgICAgICAgICByb3cgPSBzZWxmLmxhYmVsc19kZltzZWxmLmxhYmVsc19kZlsi")
parts.append("U3R1ZHlJbnN0YW5jZVVJRCJdID09IHVpZF0NCiAgICAgICAgICAgICAgICBpZiBsZW4ocm93KToNCiAgICAgICAgICAgICAgICAgICAgZm9yIGosIGMgaW4gZW51bWVyYXRlKExBQkVMUyk6DQogICAgICAgICAgICAgICAgICAgICAgICB2ID0gcm93Lmlsb2NbMF1b")
parts.append("Y10NCiAgICAgICAgICAgICAgICAgICAgICAgIGlmIHBkLm5vdG5hKHYpOg0KICAgICAgICAgICAgICAgICAgICAgICAgICAgIHlbal0gPSBmbG9hdCh2KQ0KICAgICAgICAgICAgICAgICAgICAgICAgICAgIHZhbGlkW2pdID0gMS4wDQogICAgICAgICAgICByZXR1")
parts.append("cm4geCwgeSwgdmFsaWQNCg0KDQogICAgIyAtLS0tLS0tLS0tLS0tLS0tIOaooeWeiyAtLS0tLS0tLS0tLS0tLS0tDQoNCiAgICBfUFJFVFJBSU5FRF9UUklFRCA9IFtGYWxzZV0gICMg5qih5Z2X57qn5qCH5b+XOuS4i+i9veWksei0pei/h+S4gOasoeWwseS4jeWG")
parts.append("jeWPjeWkjemHjeivlQ0KDQoNCiAgICBkZWYgX2J1aWxkX3Jlc25ldDM0KHVzZV9wcmV0cmFpbmVkPUZhbHNlLCBvZmZsaW5lX3BhdGhzPU5vbmUpOg0KICAgICAgICAiIiLmnoTpgKAgUmVzTmV0MzQg5bm25Yqg6L29IEltYWdlTmV0IOmihOiuree7g+adg+mHjeOA")
parts.append("gg0KDQogICAgICAgIOadg+mHjeadpea6kOS8mOWFiOe6pzoNCiAgICAgICAgMS4gb2ZmbGluZV9wYXRoczrmjILovb3nmoTnprvnur/mnYPph43mlofku7Yo5omr5o+PIC9rYWdnbGUvaW5wdXQg5b6X5YiwLOaXoOmcgOiBlOe9kSk7DQogICAgICAgIDIuIHVzZV9w")
parts.append("cmV0cmFpbmVkPVRydWU66IGU572R5LiL6L29IHRvcmNodmlzaW9uIOadg+mHjSjluKbph43or5Us5aSx6LSl5Zue6YCA6ZqP5py6KTsNCiAgICAgICAgMy4g5ZCm5YiZ6ZqP5py65Yid5aeL5YyW44CCDQogICAgICAgIOaIkOWKn+WKoOi9veWQjuadg+mHjeeUsSB0")
parts.append("b3JjaCDnvJPlrZgs5ZCO57utIGZvbGQg56eS5Yqg6L2944CCDQogICAgICAgICIiIg0KICAgICAgICBuZXQgPSB0dm0ucmVzbmV0MzQod2VpZ2h0cz1Ob25lKQ0KICAgICAgICBpZiBvZmZsaW5lX3BhdGhzOg0KICAgICAgICAgICAgb2ssIGRlc2MgPSBfdHJ5X2xv")
parts.append("YWRfb2ZmbGluZShuZXQsIG9mZmxpbmVfcGF0aHMpDQogICAgICAgICAgICBpZiBvazoNCiAgICAgICAgICAgICAgICByZXR1cm4gbmV0DQogICAgICAgICAgICBwcmludChmIlvmnYPph41dIOemu+e6v+WAmemAieWdh+S4jeWMuemFjSh7ZGVzY30pLOWbnumAgOWI")
parts.append("sOmaj+acui/ogZTnvZHot6/lvoQiLCBmbHVzaD1UcnVlKQ0KICAgICAgICBpZiBub3QgdXNlX3ByZXRyYWluZWQ6DQogICAgICAgICAgICBwcmludCgiW+adg+mHjV0g5pyq5byA5ZCvIC0tcHJldHJhaW5lZCDkuJTml6Dnprvnur/mnYPph40s5L2/55So6ZqP5py6")
parts.append("5Yid5aeL5YyWIg0KICAgICAgICAgICAgICAgICAgIijkuI3ogZTnvZE75aaC6ZyA6aKE6K6t57uD6K+35YqgIC0tcHJldHJhaW5lZCDlubblvIDlkK8gSW50ZXJuZXQsIg0KICAgICAgICAgICAgICAgICAgIuaIluaMgui9veemu+e6v+adg+mHjeaVsOaNrumbhiki")
parts.append("LCBmbHVzaD1UcnVlKQ0KICAgICAgICAgICAgcmV0dXJuIG5ldA0KICAgICAgICBpZiBfUFJFVFJBSU5FRF9UUklFRFswXToNCiAgICAgICAgICAgIHJldHVybiBuZXQgICMg5LmL5YmN5bey56Gu6K6k5LiL6L295LiN5LqGLOebtOaOpemaj+acug0KICAgICAgICBm")
parts.append("b3IgYXR0ZW1wdCBpbiByYW5nZSgxLCA0KToNCiAgICAgICAgICAgIHRyeToNCiAgICAgICAgICAgICAgICBuZXQgPSB0dm0ucmVzbmV0MzQod2VpZ2h0cz10dm0uUmVzTmV0MzRfV2VpZ2h0cy5JTUFHRU5FVDFLX1YxDQogICAgICAgICAgICAgICAgICAgICAgICAg")
parts.append("ICAgICAgICAgIGlmIGhhc2F0dHIodHZtLCAiUmVzTmV0MzRfV2VpZ2h0cyIpIGVsc2UgTm9uZSkNCiAgICAgICAgICAgICAgICByZXR1cm4gbmV0DQogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6DQogICAgICAgICAgICAgICAgcHJpbnQoZiJb5p2D")
parts.append("6YeNXSBSZXNOZXQzNCDpooTorq3nu4PkuIvovb3lpLHotKUo56ysIHthdHRlbXB0fS8zIOasoSk6IHtlfSIsIGZsdXNoPVRydWUpDQogICAgICAgICAgICAgICAgdGltZS5zbGVlcCg1KQ0KICAgICAgICBfUFJFVFJBSU5FRF9UUklFRFswXSA9IFRydWUNCiAgICAg")
parts.append("ICAgcHJpbnQoIlvmnYPph41dIDMg5qyh5Z2H5aSx6LSlLOaUueeUqOmaj+acuuWIneWni+WMlijnsr7luqbkvJrmmI7mmL7kuIvpmY075bu66K6u6YeN6K+V5oiW5oyC6L296aKE6K6t57uD5p2D6YeN5pWw5o2u6ZuGKSIsIGZsdXNoPVRydWUpDQogICAgICAgIHJl")
parts.append("dHVybiB0dm0ucmVzbmV0MzQod2VpZ2h0cz1Ob25lKQ0KDQoNCiAgICBjbGFzcyBLbmVlTW9kZWwobm4uTW9kdWxlKToNCiAgICAgICAgIiIiUmVzTmV0MzQg5YWx5LqrIGJhY2tib25lICsg5YWo5bGA5rGg5YyWICsgMTIgc2lnbW9pZCDlpLQo5q+P5Liq5bmz6Z2i")
parts.append("5LiA5Liq5a6e5L6LKeOAgiIiIg0KDQogICAgICAgIGRlZiBfX2luaXRfXyhzZWxmLCBudW1fbGFiZWxzPWxlbihMQUJFTFMpLCBjaGFubmVscz1DTElQLCBwcmV0cmFpbmVkPUZhbHNlLCBvZmZsaW5lX3BhdGhzPU5vbmUpOg0KICAgICAgICAgICAgc3VwZXIoKS5f")
parts.append("X2luaXRfXygpDQogICAgICAgICAgICBpZiBIQVNfVFY6DQogICAgICAgICAgICAgICAgc2VsZi5iYWNrYm9uZSA9IF9idWlsZF9yZXNuZXQzNCh1c2VfcHJldHJhaW5lZD1wcmV0cmFpbmVkLA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAg")
parts.append("ICAgICAgICAgb2ZmbGluZV9wYXRocz1vZmZsaW5lX3BhdGhzKQ0KICAgICAgICAgICAgZWxzZToNCiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIumcgOimgSB0b3JjaHZpc2lvbijor7flnKggS2FnZ2xlIE5vdGVib29rIEdQVSDnjq/looPov5Do")
parts.append("oYwpIikNCiAgICAgICAgICAgIGNvbnYxX3cgPSBzZWxmLmJhY2tib25lLmNvbnYxLndlaWdodCAgIyDkv53nlZnpooTorq3nu4MgY29udjEg5p2D6YeNDQogICAgICAgICAgICBzZWxmLmJhY2tib25lLmNvbnYxID0gbm4uQ29udjJkKGNoYW5uZWxzLCA2NCwga2Vy")
parts.append("bmVsX3NpemU9Nywgc3RyaWRlPTIsIHBhZGRpbmc9MywgYmlhcz1GYWxzZSkNCiAgICAgICAgICAgIGlmIGNvbnYxX3cuc2hhcGUgPT0gc2VsZi5iYWNrYm9uZS5jb252MS53ZWlnaHQuc2hhcGU6ICAjIGNoYW5uZWxzPT0zIOaXtuebtOaOpeayv+eUqA0KICAgICAg")
parts.append("ICAgICAgICAgIHNlbGYuYmFja2JvbmUuY29udjEud2VpZ2h0LmRhdGEuY29weV8oY29udjFfdy5kYXRhKQ0KICAgICAgICAgICAgZWxpZiBjaGFubmVscyAlIDMgPT0gMCBhbmQgY29udjFfdy5zaGFwZVsxXSA9PSAzOg0KICAgICAgICAgICAgICAgICMgMi41RCDp")
parts.append("gJrpgZPmianlsZU65oqKIDMg6YCa6YGT6aKE6K6t57uD5p2D6YeN5aSN5Yi2IG5fY2xpcHMg5Lu9KOW4uOingeWBmuazlSkNCiAgICAgICAgICAgICAgICByZXAgPSBjaGFubmVscyAvLyAzDQogICAgICAgICAgICAgICAgc2VsZi5iYWNrYm9uZS5jb252MS53ZWln")
parts.append("aHQuZGF0YS5jb3B5Xyhjb252MV93LnJlcGVhdCgxLCByZXAsIDEsIDEpKQ0KICAgICAgICAgICAgICAgIHByaW50KGYiW+adg+mHjV0gY29udjEg6aKE6K6t57uD5p2D6YeN5aSN5Yi2IHtyZXB9IOS7veaJqeWxleWIsCB7Y2hhbm5lbHN9IOmAmumBkyIsIGZsdXNo")
parts.append("PVRydWUpDQogICAgICAgICAgICBzZWxmLmJhY2tib25lLmZjID0gbm4uSWRlbnRpdHkoKQ0KICAgICAgICAgICAgc2VsZi5oZWFkID0gbm4uTGluZWFyKDUxMiwgbnVtX2xhYmVscykNCg0KICAgICAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToNCiAgICAgICAgICAg")
parts.append("IGZlYXQgPSBzZWxmLmJhY2tib25lKHgpICAjIChCLCA1MTIpDQogICAgICAgICAgICByZXR1cm4gc2VsZi5oZWFkKGZlYXQpICAgIyBsb2dpdHMgKEIsIDEyKQ0KDQoNCg0KIyAtLS0tLS0tLS0tLS0tLS0tIOiuree7gyAvIOmqjOivgSAtLS0tLS0tLS0tLS0tLS0t")
parts.append("DQoNCmRlZiB0cmFpbl9lcG9jaChtb2RlbCwgbG9hZGVyLCBvcHQsIGNyaXQsIGRldmljZSk6DQogICAgbW9kZWwudHJhaW4oKQ0KICAgIHRvdCwgY250ID0gMC4wLCAwDQogICAgZm9yIHgsIHksIHZhbGlkIGluIGxvYWRlcjoNCiAgICAgICAgeCwgeSwgdmFsaWQg")
parts.append("PSB4LnRvKGRldmljZSksIHkudG8oZGV2aWNlKSwgdmFsaWQudG8oZGV2aWNlKQ0KICAgICAgICBvcHQuemVyb19ncmFkKCkNCiAgICAgICAgb3V0ID0gbW9kZWwoeCkNCiAgICAgICAgaWYgdmFsaWQuc3VtKCkgPT0gMDoNCiAgICAgICAgICAgIGNvbnRpbnVlDQog")
parts.append("ICAgICAgIGxvc3MgPSAoY3JpdChvdXQsIHkpICogdmFsaWQpLnN1bSgpIC8gdmFsaWQuc3VtKCkNCiAgICAgICAgbG9zcy5iYWNrd2FyZCgpDQogICAgICAgIG9wdC5zdGVwKCkNCiAgICAgICAgdG90ICs9IGxvc3MuaXRlbSgpICogbGVuKHgpDQogICAgICAgIGNu")
parts.append("dCArPSBsZW4oeCkNCiAgICByZXR1cm4gdG90IC8gbWF4KGNudCwgMSkNCg0KDQpkZWYgcHJlZGljdChtb2RlbCwgbG9hZGVyLCBkZXZpY2UpOg0KICAgIG1vZGVsLmV2YWwoKQ0KICAgIHJvd3MgPSBbXQ0KICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOg0KICAgICAg")
parts.append("ICBmb3IgeCwgXywgXyBpbiBsb2FkZXI6DQogICAgICAgICAgICB4ID0geC50byhkZXZpY2UpDQogICAgICAgICAgICBvdXQgPSB0b3JjaC5zaWdtb2lkKG1vZGVsKHgpKQ0KICAgICAgICAgICAgIyAuY3B1KCkudG9saXN0KCkg6ICM6Z2eIC5jcHUoKS5udW1weSgp")
parts.append("OnRvcmNoIDIuMi4wK2N1MTE4IOe8luS6jiBudW1weTEueCwNCiAgICAgICAgICAgICMg5ZyoIEthZ2dsZSBudW1weTIuMC4yIOS4iyB0b3JjaOKGkm51bXB5IGludGVyb3Ag5bSpO3RvbGlzdCDnuq8gQysrIOi3r+W+hOWuieWFqOOAgg0KICAgICAgICAgICAgcm93")
parts.append("cy5leHRlbmQob3V0LmNwdSgpLnRvbGlzdCgpKQ0KICAgIHJldHVybiBucC5hc2FycmF5KHJvd3MsIGR0eXBlPW5wLmZsb2F0MzIpIGlmIHJvd3MgZWxzZSBucC56ZXJvcygoMCwgbGVuKExBQkVMUykpLCBkdHlwZT1ucC5mbG9hdDMyKQ0KDQoNCmRlZiBydW5fZm9s")
parts.append("ZChtb2RlbCwgdHJhaW5fZHMsIHZhbF9kcywgZGV2aWNlLCBhcmdzKToNCiAgICB0cmFpbl9sb2FkZXIgPSBEYXRhTG9hZGVyKHRyYWluX2RzLCBiYXRjaF9zaXplPWFyZ3MuYmF0Y2gsIHNodWZmbGU9VHJ1ZSwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAg")
parts.append("IG51bV93b3JrZXJzPWFyZ3Mud29ya2VycywgZHJvcF9sYXN0PUZhbHNlKQ0KICAgIHZhbF9sb2FkZXIgPSBEYXRhTG9hZGVyKHZhbF9kcywgYmF0Y2hfc2l6ZT1hcmdzLmJhdGNoLCBzaHVmZmxlPUZhbHNlLA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgIG51")
parts.append("bV93b3JrZXJzPWFyZ3Mud29ya2VycywgZHJvcF9sYXN0PUZhbHNlKQ0KICAgIG9wdCA9IEFkYW1XKG1vZGVsLnBhcmFtZXRlcnMoKSwgbHI9YXJncy5sciwgd2VpZ2h0X2RlY2F5PTFlLTQpDQogICAgc2NoZWQgPSBDb3NpbmVBbm5lYWxpbmdMUihvcHQsIFRfbWF4")
parts.append("PWFyZ3MuZXBvY2hzKQ0KICAgIGNyaXQgPSBubi5CQ0VXaXRoTG9naXRzTG9zcyhyZWR1Y3Rpb249Im5vbmUiKQ0KICAgIGZvciBlcCBpbiByYW5nZShhcmdzLmVwb2Nocyk6DQogICAgICAgIHQwID0gdGltZS50aW1lKCkNCiAgICAgICAgbG9zcyA9IHRyYWluX2Vw")
parts.append("b2NoKG1vZGVsLCB0cmFpbl9sb2FkZXIsIG9wdCwgY3JpdCwgZGV2aWNlKQ0KICAgICAgICBzY2hlZC5zdGVwKCkNCiAgICAgICAgaWYgYXJncy52ZXJib3NlOg0KICAgICAgICAgICAgcHJpbnQoZiIgIGVwb2NoIHtlcCArIDF9L3thcmdzLmVwb2Noc30gbG9zcyB7")
parts.append("bG9zczouNGZ9ICh7dGltZS50aW1lKCkgLSB0MDouMGZ9cykiLCBmbHVzaD1UcnVlKQ0KICAgIHZhbF9wcmVkID0gcHJlZGljdChtb2RlbCwgdmFsX2xvYWRlciwgZGV2aWNlKQ0KICAgICMgRElDT00g5ZG95Lit6K+K5patOuiuree7gyvpqozor4Hlt7Lov63ku6Pl")
parts.append("rozmr5Us5q2k5pe26K6h5pWw5omN5YeG56GuDQogICAgdGgsIGVoLCBhdCA9IHRyYWluX2RzLnN0YXRzKCkNCiAgICB2aCwgZXYsIGF2ID0gdmFsX2RzLnN0YXRzKCkNCiAgICBwcmludChmIlvmlbDmja5dIHRyYWluIOWRveS4rSB7dGh9L3tsZW4odHJhaW5fZHMp")
parts.append("fSBzdHVkeSjnqboge2VofSwg5bmz5Z2HIHthdDouMWZ9IGNsaXApIHwgIg0KICAgICAgICAgIGYidmFsIOWRveS4rSB7dmh9L3tsZW4odmFsX2RzKX0gc3R1ZHko56m6IHtldn0pIiwgZmx1c2g9VHJ1ZSkNCiAgICBpZiB0aCA9PSAwIG9yIHZoID09IDA6DQogICAg")
parts.append("ICAgIHByaW50KCJb6K2m5ZGKXSDmnInmipggRElDT00g5YWo6YOo5rKh6K+75YiwKOWRveS4rSAwKeKAlOKAlOi+k+WFpeWwhuaYr+WFqOmbtizliIbmlbDlv4XnhLYgfjAuNSEiLCBmbHVzaD1UcnVlKQ0KICAgIHZhbF91aWQgPSBucC5hcnJheSh2YWxfZHMuc3R1")
parts.append("ZGllcykgICMgc2h1ZmZsZT1GYWxzZSzpobrluo/kuI7mlbDmja7pm4bkuIDoh7QNCiAgICByZXR1cm4gdmFsX3ByZWQsIHZhbF91aWQNCg0KDQojIC0tLS0tLS0tLS0tLS0tLS0g5o6o55CGIC8g6J6N5ZCIIC8g5o+Q5LqkIC0tLS0tLS0tLS0tLS0tLS0NCg0KZGVm")
parts.append("IHByZWRpY3RfdGVzdChtb2RlbCwgdGVzdF9kZiwgc2VyaWVzX2RmLCByb290LCBkZXZpY2UsIHBsYW5lLCBuX2NsaXBzPTIpOg0KICAgICIiIuWvuSB0ZXN0IOavj+S4qiBzdHVkeSDmjInmjIflrprlubPpnaLmlLbpm4YgY2xpcHMs6aKE5rWLIDEyIOS4quamgueO")
parts.append("h+OAgiIiIg0KICAgIGRzID0gS25lZURhdGFzZXQodGVzdF9kZlsiU3R1ZHlJbnN0YW5jZVVJRCJdLnRvbGlzdCgpLCBOb25lLCByb290LCBzZXJpZXNfZGYsDQogICAgICAgICAgICAgICAgICAgICBwbGFuZSwgaGFzX3RhcmdldHM9RmFsc2UsIG5fY2xpcHM9bl9j")
parts.append("bGlwcykNCiAgICBsb2FkZXIgPSBEYXRhTG9hZGVyKGRzLCBiYXRjaF9zaXplPTE2LCBzaHVmZmxlPUZhbHNlLCBudW1fd29ya2Vycz0wKQ0KICAgIHJldHVybiBwcmVkaWN0KG1vZGVsLCBsb2FkZXIsIGRldmljZSkNCg0KDQpkZWYgcGxhbmVfb29mX2F1Yyhvb2Ys")
parts.append("IGxhYmVsc19kZiwgdWlkX2xpc3QpOg0KICAgICIiIuiuoeeul+S4gOS4quW5s+mdouaooeWei+WcqOaMh+WumiBzdHVkeSDlrZDpm4bkuIrnmoTlro/lubPlnYcgT09GIEFVQyjlj6rnrpfmnInmoIfms6jnmoTliJcp44CCDQoNCiAgICBsYWJlbHNfZGYg5b+F6aG7")
parts.append("5piv44CQ5a6Y5pa5IGdvbGQg5Y6f5aeL6KGo44CRKDAvMS9OYU4s5aaCIHRyYWluX2NzdiDljp/liJcpLOS4jeiDveS8oA0KICAgIGxvYWRfbGFiZWxzIOWhq+WFheWQjueahCBzb2Z0IOihqOKAlOKAlHNvZnQg5Lyq5qCH562+5pei6Z2eIGdyb3VuZCB0cnV0aCzk")
parts.append("uZ/kvJrorqkNCiAgICByb2NfYXVjX3Njb3JlIOW0qShjb250aW51b3VzKeOAgjU4IOS+i+Wwj+agt+acrOS4i+afkOWIl+WPr+iDveWPquacieWNleS4gOexu+WIqyzot7Pov4for6XliJfjgIINCiAgICAiIiINCiAgICBhdWNzID0gW10NCiAgICBmb3IgaiwgYyBp")
parts.append("biBlbnVtZXJhdGUoTEFCRUxTKToNCiAgICAgICAgeV90cnVlID0gbGFiZWxzX2RmLnNldF9pbmRleCgiU3R1ZHlJbnN0YW5jZVVJRCIpW2NdLnJlaW5kZXgodWlkX2xpc3QpLnZhbHVlcy5hc3R5cGUoZmxvYXQpDQogICAgICAgIHZhbGlkID0gfm5wLmlzbmFuKHlf")
parts.append("dHJ1ZSkNCiAgICAgICAgaWYgdmFsaWQuc3VtKCkgPj0gMjA6DQogICAgICAgICAgICB5diA9IHlfdHJ1ZVt2YWxpZF0NCiAgICAgICAgICAgIGlmIGxlbihucC51bmlxdWUoeXYpKSA8IDI6DQogICAgICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgICAgIGF1")
parts.append("Y3MuYXBwZW5kKHJvY19hdWNfc2NvcmUoeXYsIG9vZlt2YWxpZCwgal0pKQ0KICAgIHJldHVybiBmbG9hdChucC5tZWFuKGF1Y3MpKSBpZiBhdWNzIGVsc2UgMC4wDQoNCg0KIyAtLS0tLS0tLS0tLS0tLS0tIGFydGlmYWN0IC8gcmVzdW1lIC8gZW5zZW1ibGUgaGVs")
parts.append("cGVycyAtLS0tLS0tLS0tLS0tLS0tDQoNCmRlZiBfYXJ0aWZhY3Rfcm9vdChhcmdzKToNCiAgICBiYXNlID0gUGF0aChnZXRhdHRyKGFyZ3MsICJhcnRpZmFjdF9kaXIiLCAiIikgb3IgImFydGlmYWN0cyIpDQogICAgZXhwID0gZ2V0YXR0cihhcmdzLCAiZXhwZXJp")
parts.append("bWVudF9pZCIsICJleHBfMDAxXzVmb2xkX3NhZ2l0dGFsIikNCiAgICByZXR1cm4gYmFzZSAvIGV4cA0KDQoNCmRlZiBfZm9sZF9tZXRhX3BhdGgoYXJ0aWZhY3Rfcm9vdCwgZm9sZCk6DQogICAgcmV0dXJuIGFydGlmYWN0X3Jvb3QgLyBmImZvbGR7Zm9sZH0iIC8g")
parts.append("Im1ldGEuanNvbiINCg0KDQpkZWYgX2lzX2ZvbGRfZG9uZShhcnRpZmFjdF9yb290LCBmb2xkKToNCiAgICBwID0gX2ZvbGRfbWV0YV9wYXRoKGFydGlmYWN0X3Jvb3QsIGZvbGQpDQogICAgaWYgbm90IHAuZXhpc3RzKCk6DQogICAgICAgIHJldHVybiBGYWxzZQ0K")
parts.append("ICAgIHRyeToNCiAgICAgICAgbWV0YSA9IGpzb24ubG9hZHMocC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpDQogICAgICAgIHJldHVybiBtZXRhLmdldCgiY29tcGxldGlvbl9zdGF0dXMiKSA9PSAiZG9uZSINCiAgICBleGNlcHQgRXhjZXB0aW9uOg0KICAg")
parts.append("ICAgICByZXR1cm4gRmFsc2UNCg0KDQpkZWYgX3NhdmVfZm9sZF9hcnRpZmFjdChhcnRpZmFjdF9yb290LCBmb2xkLCB0ZXN0X3ByZWQsIHRlc3RfdWlkcywgbWV0YSk6DQogICAgZm9sZF9kaXIgPSBhcnRpZmFjdF9yb290IC8gZiJmb2xke2ZvbGR9Ig0KICAgIGZv")
parts.append("bGRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkNCiAgICBucC5zYXZlKHN0cihmb2xkX2RpciAvICJ0ZXN0X3ByZWQubnB5IiksIHRlc3RfcHJlZC5hc3R5cGUobnAuZmxvYXQzMikpDQogICAgcGQuRGF0YUZyYW1lKHsiU3R1ZHlJbnN0YW5j")
parts.append("ZVVJRCI6IHRlc3RfdWlkc30pLnRvX2NzdigNCiAgICAgICAgZm9sZF9kaXIgLyAidGVzdF91aWQuY3N2IiwgaW5kZXg9RmFsc2UNCiAgICApDQogICAgbWV0YVsiY29tcGxldGlvbl9zdGF0dXMiXSA9ICJkb25lIg0KICAgIG1ldGFbInByZWRpY3Rpb25fc2hhcGUi")
parts.append("XSA9IGxpc3QodGVzdF9wcmVkLnNoYXBlKQ0KICAgIChmb2xkX2RpciAvICJtZXRhLmpzb24iKS53cml0ZV90ZXh0KA0KICAgICAgICBqc29uLmR1bXBzKG1ldGEsIGVuc3VyZV9hc2NpaT1GYWxzZSwgaW5kZW50PTIpLCBlbmNvZGluZz0idXRmLTgiDQogICAgKQ0K")
parts.append("DQoNCmRlZiBfbG9hZF9mb2xkX3ByZWRpY3Rpb25zKGFydGlmYWN0X3Jvb3QsIGZvbGQsIGNhbm9uaWNhbF91aWRzKToNCiAgICBmb2xkX2RpciA9IGFydGlmYWN0X3Jvb3QgLyBmImZvbGR7Zm9sZH0iDQogICAgcHJlZCA9IG5wLmxvYWQoZm9sZF9kaXIgLyAidGVz")
parts.append("dF9wcmVkLm5weSIpDQogICAgdWlkcyA9IHBkLnJlYWRfY3N2KGZvbGRfZGlyIC8gInRlc3RfdWlkLmNzdiIpWyJTdHVkeUluc3RhbmNlVUlEIl0udG9saXN0KCkNCiAgICBpZiBwcmVkLnNoYXBlICE9IChsZW4oY2Fub25pY2FsX3VpZHMpLCBsZW4oTEFCRUxTKSk6")
parts.append("DQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJmb2xke2ZvbGR9IHNoYXBlIG1pc21hdGNoOiB7cHJlZC5zaGFwZX0gdnMgZXhwZWN0ZWQgeyhsZW4oY2Fub25pY2FsX3VpZHMpLCBsZW4oTEFCRUxTKSl9IikNCiAgICBpZiB1aWRzICE9IGNhbm9uaWNhbF91aWRz")
parts.append("Og0KICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiZm9sZHtmb2xkfSBVSUQgb3JkZXIgbWlzbWF0Y2giKQ0KICAgIGlmIG5vdCBucC5pc2Zpbml0ZShwcmVkKS5hbGwoKToNCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmImZvbGR7Zm9sZH0gY29udGFpbnMgTmFO")
parts.append("L0luZiIpDQogICAgaWYgcHJlZC5taW4oKSA8IDAgb3IgcHJlZC5tYXgoKSA+IDE6DQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJmb2xke2ZvbGR9IHByZWRpY3Rpb25zIG91dCBvZiBbMCwxXSIpDQogICAgcmV0dXJuIHByZWQNCg0KDQpkZWYgX3NhdmVfZXhw")
parts.append("ZXJpbWVudF9tZXRhKGFydGlmYWN0X3Jvb3QsIHRyYWluX2ZvbGRfdWlkcywgdGVzdF91aWRzLCBhcmdzKToNCiAgICBtZXRhID0gew0KICAgICAgICAiZXhwZXJpbWVudF9pZCI6IGdldGF0dHIoYXJncywgImV4cGVyaW1lbnRfaWQiLCAiZXhwXzAwMV81Zm9sZF9z")
parts.append("YWdpdHRhbCIpLA0KICAgICAgICAiZ2l0X2NvbW1pdCI6IF9fZ2l0X2NvbW1pdCgpLA0KICAgICAgICAiYXJncyI6IHZhcnMoYXJncyksDQogICAgICAgICJsYWJlbHMiOiBMQUJFTFMsDQogICAgICAgICJ0cmFpbl9mb2xkMF91aWRzIjogdHJhaW5fZm9sZF91aWRz")
parts.append("LA0KICAgICAgICAidGVzdF91aWRzIjogdGVzdF91aWRzLA0KICAgIH0NCiAgICAoYXJ0aWZhY3Rfcm9vdCAvICJjb25maWcuanNvbiIpLndyaXRlX3RleHQoDQogICAgICAgIGpzb24uZHVtcHMobWV0YSwgZW5zdXJlX2FzY2lpPUZhbHNlLCBpbmRlbnQ9MiksIGVu")
parts.append("Y29kaW5nPSJ1dGYtOCINCiAgICApDQogICAgcGQuRGF0YUZyYW1lKHsiU3R1ZHlJbnN0YW5jZVVJRCI6IHRlc3RfdWlkc30pLnRvX2NzdigNCiAgICAgICAgYXJ0aWZhY3Rfcm9vdCAvICJ0ZXN0X3VpZC5jc3YiLCBpbmRleD1GYWxzZQ0KICAgICkNCiAgICBmb2xk")
parts.append("X3Jvd3MgPSBbXQ0KICAgIGZvciBrIGluIHJhbmdlKGFyZ3MuZm9sZHMpOg0KICAgICAgICBmb2xkX3Jvd3MuYXBwZW5kKHsiZm9sZCI6IGssICJ0cmFpbl91aWRzIjogdHJhaW5fZm9sZF91aWRzW2tdfSkNCiAgICBwZC5EYXRhRnJhbWUoZm9sZF9yb3dzKS50b19j")
parts.append("c3YoYXJ0aWZhY3Rfcm9vdCAvICJmb2xkX2Fzc2lnbm1lbnQuY3N2IiwgaW5kZXg9RmFsc2UpDQoNCg0KZGVmIF9tYXliZV9rYWdnbGVfZGF0YXNldF92ZXJzaW9uKGFydGlmYWN0X3Jvb3QsIG1lc3NhZ2UpOg0KICAgIGlmIG9zLmVudmlyb24uZ2V0KCJLQUdHTEVf")
parts.append("S0VSTkVMX1JVTl9UWVBFIikgIT0gIkludGVyYWN0aXZlIjoNCiAgICAgICAgcmV0dXJuDQogICAgc2x1ZyA9IG9zLmVudmlyb24uZ2V0KCJLQUdHTEVfS0VSTkVMX1NMVUciLCAicnNuYS1hcnRpZmFjdHMtdG1wIikNCiAgICBvd25lciA9IG9zLmVudmlyb24uZ2V0")
parts.append("KCJLQUdHTEVfVVNFUk5BTUUiLCAieml5aW1pbmciKQ0KICAgIGRhdGFzZXRfaWQgPSBmIntvd25lcn0ve3NsdWd9Ig0KICAgIHRyeToNCiAgICAgICAgc3VicHJvY2Vzcy5ydW4oDQogICAgICAgICAgICBbDQogICAgICAgICAgICAgICAgImthZ2dsZSIsDQogICAg")
parts.append("ICAgICAgICAgICAgImRhdGFzZXRzIiwNCiAgICAgICAgICAgICAgICAidmVyc2lvbiIsDQogICAgICAgICAgICAgICAgIi1wIiwNCiAgICAgICAgICAgICAgICBzdHIoYXJ0aWZhY3Rfcm9vdCksDQogICAgICAgICAgICAgICAgIi1tIiwNCiAgICAgICAgICAgICAg")
parts.append("ICBtZXNzYWdlLA0KICAgICAgICAgICAgICAgICItLWRpci1tb2RlIiwNCiAgICAgICAgICAgICAgICAidGFyIiwNCiAgICAgICAgICAgIF0sDQogICAgICAgICAgICBjaGVjaz1UcnVlLA0KICAgICAgICAgICAgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwNCiAgICAgICAg")
parts.append("KQ0KICAgICAgICBwcmludChmIlthcnRpZmFjdF0gdXBsb2FkZWQgZGF0YXNldCB2ZXJzaW9uOiB7ZGF0YXNldF9pZH0iLCBmbHVzaD1UcnVlKQ0KICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToNCiAgICAgICAgcHJpbnQoZiJbYXJ0aWZhY3RdIGRhdGFzZXQgdXBs")
parts.append("b2FkIHNraXBwZWQ6IHtlfSIsIGZsdXNoPVRydWUpDQoNCg0KZGVmIF9fZ2l0X2NvbW1pdCgpOg0KICAgIHRyeToNCiAgICAgICAgaW1wb3J0IHN1YnByb2Nlc3MNCiAgICAgICAgcmV0dXJuIHN1YnByb2Nlc3MuY2hlY2tfb3V0cHV0KA0KICAgICAgICAgICAgWyJn")
parts.append("aXQiLCAicmV2LXBhcnNlIiwgIkhFQUQiXSwgdGV4dD1UcnVlDQogICAgICAgICkuc3RyaXAoKQ0KICAgIGV4Y2VwdCBFeGNlcHRpb246DQogICAgICAgIHJldHVybiAidW5rbm93biINCg0KDQpkZWYgbWFpbigpOg0KICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQ")
parts.append("YXJzZXIoKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgiLS1lcG9jaHMiLCB0eXBlPWludCwgZGVmYXVsdD0yKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgiLS1iYXRjaCIsIHR5cGU9aW50LCBkZWZhdWx0PTgpDQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWxyIiwgdHlwZT1m")
parts.append("bG9hdCwgZGVmYXVsdD0zZS00KQ0KICAgIGFwLmFkZF9hcmd1bWVudCgiLS13b3JrZXJzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MCkNCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbi1jbGlwcyIsIHR5cGU9aW50LCBkZWZhdWx0PTIpDQogICAgYXAuYWRkX2FyZ3VtZW50")
parts.append("KCItLXNlZWQiLCB0eXBlPWludCwgZGVmYXVsdD1TRUVEKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgiLS1mb2xkcyIsIHR5cGU9aW50LCBkZWZhdWx0PUZPTERTKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgiLS1vbmx5LWZvbGQiLCB0eXBlPWludCwgZGVmYXVsdD0tMSwN")
parts.append("CiAgICAgICAgICAgICAgICAgICAgaGVscD0i5Y+q6LeR5oyH5a6a5oqYKOWFtuS9mei3s+i/hzvpu5jorqQgLTE95YWo6YOoKeOAguaOoui3r+eUqDotLWZvbGRzIDUgLS1vbmx5LWZvbGQgMCDorq3nrKwgMCDmipgiDQogICAgICAgICAgICAgICAgICAgICAgICAg")
parts.append("Iih0cmFpbiA0LzUg5LiO5q2j5byP5ZCM5YiG5biDKSx0ZXN0IOaOqOeQhueUqOivpeaKmOaooeWei+WHuuaPkOS6pCIpDQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXBsYW5lcyIsIHR5cGU9c3RyLCBkZWZhdWx0PSIsIi5qb2luKFBMQU5FUyksDQogICAgICAgICAg")
parts.append("ICAgICAgICAgIGhlbHA9IumAl+WPt+WIhumalOeahOW5s+mdouWtkOmbhizlpoIgYXhpYWwsc2FnaXR0YWwo5bCP5YaZLOS4jeWMuuWIhuWkp+Wwj+WGmSkiKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgiLS1mdXNlIiwgdHlwZT1zdHIsIGRlZmF1bHQ9IndlaWdodCIs")
parts.append("DQogICAgICAgICAgICAgICAgICAgIGNob2ljZXM9WyJ3ZWlnaHQiLCAibWVhbiJdLA0KICAgICAgICAgICAgICAgICAgICBoZWxwPSLlubPpnaLono3lkIjmlrnlvI86d2VpZ2h0PeaMiSBPT0YgQVVDIOWKoOadgzttZWFuPeeugOWNleW5s+WdhyIpDQogICAgYXAu")
parts.append("YWRkX2FyZ3VtZW50KCItLXBzZXVkby1jb25mIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0wLjAsDQogICAgICAgICAgICAgICAgICAgIGhlbHA9IuS8quagh+etvue9ruS/oeW6pui/h+a7pOmYiOWAvCgwPeS4jei/h+a7pDswLjU95Lii5byD572u5L+h5bqmPDAuNeea")
parts.append("hOS8quagh+etvikiKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgiLS1uby1yZXBvcnQtbGFiZWxzIiwgYWN0aW9uPSJzdG9yZV90cnVlIiwNCiAgICAgICAgICAgICAgICAgICAgaGVscD0i56aB55So5YaF572u5oql5ZGK5YWz6ZSu6K+N6L2v5qCH562+5YWc5bqVKOm7")
parts.append("mOiupOW8gOWQrzrmnKrmjILnpL7ljLrmlbDmja7pm4bml7YiDQogICAgICAgICAgICAgICAgICAgICAgICAgIuiHquWKqOS7jiA0NDA3IOS7veaUvuWwhOaKpeWRiueUn+aIkOi9r+agh+etvizpm7bmjILovb0pIikNCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tdmVy")
parts.append("Ym9zZSIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpDQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXByZXRyYWluZWQiLCBhY3Rpb249InN0b3JlX3RydWUiLA0KICAgICAgICAgICAgICAgICAgICBoZWxwPSLlsJ3or5XkuIvovb0gSW1hZ2VOZXQg6aKE6K6t57uD5p2D6YeN")
parts.append("KOmcgOW8gOWQryBJbnRlcm5ldCk76buY6K6k6ZqP5py65Yid5aeL5YyW5LiN6IGU572RIikNCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tb2ZmbGluZS13ZWlnaHRzIiwgdHlwZT1zdHIsIGRlZmF1bHQ9IiIsDQogICAgICAgICAgICAgICAgICAgIGhlbHA9IumAl+WP")
parts.append("t+WIhumalOeahOemu+e6v+adg+mHjSAucHRoIOi3r+W+hCjkvJjlhYjnuqfmnIDpq5gpO+eVmeepuuWImeiHquWKqOaJq+aPj+aMgui9veebruW9lSIpDQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW5vLW9mZmxpbmUtc2NhbiIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIs")
parts.append("DQogICAgICAgICAgICAgICAgICAgIGhlbHA9IuemgeeUqOiHquWKqOaJq+aPjyAva2FnZ2xlL2lucHV0IOS4i+eahOemu+e6v+adg+mHjeaWh+S7tiIpDQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWRhdGEiLCB0eXBlPXN0ciwgZGVmYXVsdD0iIiwNCiAgICAgICAg")
parts.append("ICAgICAgICAgICAgaGVscD0i5pi+5byP5oyH5a6a5pWw5o2u5qC555uu5b2VKOWmgiBrYWdnbGVodWIuY29tcGV0aXRpb25fZG93bmxvYWQg6L+U5Zue55qEIHBhdGgpLCINCiAgICAgICAgICAgICAgICAgICAgICAgICAi5oyH5a6a5ZCO6Lez6L+H6Ieq5Yqo5p+l")
parts.append("5om+IikNCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tYXJ0aWZhY3QtZGlyIiwgdHlwZT1zdHIsIGRlZmF1bHQ9ImFydGlmYWN0cyIsDQogICAgICAgICAgICAgICAgICAgIGhlbHA9ImFydGlmYWN0IOagueebruW9lSjpu5jorqQgYXJ0aWZhY3RzKSIpDQogICAgYXAu")
parts.append("YWRkX2FyZ3VtZW50KCItLWV4cGVyaW1lbnQtaWQiLCB0eXBlPXN0ciwgZGVmYXVsdD0iZXhwXzAwMV81Zm9sZF9zYWdpdHRhbCIsDQogICAgICAgICAgICAgICAgICAgIGhlbHA9ImV4cGVyaW1lbnQgSUQo55So5LqOIGFydGlmYWN0IOWtkOebruW9lSkiKQ0KICAg")
parts.append("IGFwLmFkZF9hcmd1bWVudCgiLS1yZXN1bWUtZGlyIiwgdHlwZT1zdHIsIGRlZmF1bHQ9IiIsDQogICAgICAgICAgICAgICAgICAgIGhlbHA9IuW3suaciSBhcnRpZmFjdCDnm67lvZUs5ZCv5Yqo5pe25omr5o+PIG1ldGEuanNvbiDot7Pov4flt7LlrozmiJAgZm9s")
parts.append("ZCIpDQogICAgYXJncywgXyA9IGFwLnBhcnNlX2tub3duX2FyZ3MoKQ0KDQogICAgcmVzdW1lX3Jvb3QgPSBQYXRoKGFyZ3MucmVzdW1lX2RpcikgaWYgYXJncy5yZXN1bWVfZGlyIGVsc2UgTm9uZQ0KICAgIGlmIHJlc3VtZV9yb290IGFuZCBhcmdzLmFydGlmYWN0")
parts.append("X2RpciBhbmQgbm90IGFyZ3MuZXhwZXJpbWVudF9pZDoNCiAgICAgICAgYXJncy5leHBlcmltZW50X2lkID0gcmVzdW1lX3Jvb3QubmFtZQ0KDQogICAgaWYgbm90IEhBU19UT1JDSCBvciBub3QgSEFTX1RWIG9yIG5vdCBIQVNfUFlESUNPTToNCiAgICAgICAgc3lz")
parts.append("LmV4aXQoIuS+nei1lue8uuWksTror7flnKggS2FnZ2xlIE5vdGVib29rKEdQVSnnjq/looPov5DooYwodG9yY2gvdG9yY2h2aXNpb24vcHlkaWNvbSk75pys5Zyw5LuF5Y+v5YGa5pWw5o2u5bGC6aqM6K+B44CCIikNCg0KICAgIG5wLnJhbmRvbS5zZWVkKGFyZ3Mu")
parts.append("c2VlZCkNCiAgICB0b3JjaC5tYW51YWxfc2VlZChhcmdzLnNlZWQpDQoNCiAgICBpZiBhcmdzLmRhdGE6DQogICAgICAgIGlmIG9zLnBhdGguZXhpc3RzKG9zLnBhdGguam9pbihhcmdzLmRhdGEsICJ0cmFpbi5jc3YiKSk6DQogICAgICAgICAgICByb290LCBvdXQg")
parts.append("PSBhcmdzLmRhdGEsIGFyZ3MuZGF0YQ0KICAgICAgICAgICAgcHJpbnQoZiJb5pWw5o2uXSDkvb/nlKggLS1kYXRhIOaMh+WumuebruW9lToge3Jvb3R9IikNCiAgICAgICAgZWxzZToNCiAgICAgICAgICAgIHN5cy5leGl0KGYiLS1kYXRhIOaMh+WumuebruW9lSB7")
parts.append("YXJncy5kYXRhfSDkuIvmsqHmnIkgdHJhaW4uY3N2LOivt+ajgOafpei3r+W+hCIpDQogICAgZWxzZToNCiAgICAgICAgcm9vdCwgb3V0ID0gcmVzb2x2ZV9wYXRocygpDQogICAgcHJpbnQoZiJb5pWw5o2uXSByb290PXtyb290fSAgb3V0PXtvdXR9IikNCg0KICAg")
parts.append("IHRyYWluX2NzdiA9IHBkLnJlYWRfY3N2KG9zLnBhdGguam9pbihyb290LCAidHJhaW4uY3N2IikpDQogICAgbGFiZWxzX2RmID0gbG9hZF9sYWJlbHMocm9vdCwgcHNldWRvX2NvbmY9YXJncy5wc2V1ZG9fY29uZiwNCiAgICAgICAgICAgICAgICAgICAgICAgICAg")
parts.append("ICB1c2VfcmVwb3J0X2xhYmVscz1ub3QgYXJncy5ub19yZXBvcnRfbGFiZWxzKQ0KICAgIHNlcmllc19kZiA9IHBkLnJlYWRfY3N2KG9zLnBhdGguam9pbihyb290LCAidHJhaW5fc2VyaWVzLmNzdiIpKQ0KICAgIHRlc3RfZGYgPSBwZC5yZWFkX2Nzdihvcy5wYXRo")
parts.append("LmpvaW4ocm9vdCwgInRlc3QuY3N2IikpDQogICAgdGVzdF9zZXJpZXMgPSBwZC5yZWFkX2Nzdihvcy5wYXRoLmpvaW4ocm9vdCwgInRlc3Rfc2VyaWVzLmNzdiIpKQ0KICAgIGRjbV90cmFpbiA9IG9zLnBhdGguam9pbihyb290LCAidHJhaW5fc2VyaWVzIikNCiAg")
parts.append("ICBkY21fdGVzdCA9IG9zLnBhdGguam9pbihyb290LCAidGVzdF9zZXJpZXMiKQ0KICAgIHByaW50KGYiW+aVsOaNrl0gdHJhaW4ge2xlbih0cmFpbl9jc3YpfSBzdHVkeSAvIHRlc3Qge2xlbih0ZXN0X2RmKX0gc3R1ZHkgLyBzZXJpZXMge2xlbihzZXJpZXNfZGYp")
parts.append("fSIpDQogICAgX3ByaW50X2RjbV90cmVlKHJvb3QsICJESUNPTSIpICAjIOaJk+WNsCB0cmFpbl9zZXJpZXMg55yf5a6e55uu5b2V57uT5p6ELOehruiupOW4g+WxgA0KDQogICAgcGxhbmVzID0gW3Auc3RyaXAoKS50aXRsZSgpIGZvciBwIGluIGFyZ3MucGxhbmVz")
parts.append("LnNwbGl0KCIsIikgaWYgcC5zdHJpcCgpXQ0KICAgIHBsYW5lcyA9IFtwIGZvciBwIGluIHBsYW5lcyBpZiBwIGluIFBMQU5FU10NCiAgICBpZiBub3QgcGxhbmVzOg0KICAgICAgICBzeXMuZXhpdChmIi0tcGxhbmVzIOaXoOaViCzlj6/pgIk6IHtQTEFORVN9IikN")
parts.append("CiAgICBwcmludChmIlvlubPpnaJdIOS9v+eUqCB7cGxhbmVzfSIpDQoNCiAgICBkZXZpY2UgPSB0b3JjaC5kZXZpY2UoImN1ZGEiIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAiY3B1IikNCiAgICBwcmludChmIlvorr7lpIddIHtkZXZpY2V9IikN")
parts.append("Cg0KICAgICMg56a757q/6aKE6K6t57uD5p2D6YeNOuaYvuW8j+aMh+WumuS8mOWFiCzlkKbliJnoh6rliqjmiavmj48gL2thZ2dsZS9pbnB1dA0KICAgIG9mZmxpbmVfcGF0aHMgPSBbXQ0KICAgIGlmIGFyZ3Mub2ZmbGluZV93ZWlnaHRzLnN0cmlwKCk6DQogICAg")
parts.append("ICAgIGZvciBwIGluIFt4LnN0cmlwKCkgZm9yIHggaW4gYXJncy5vZmZsaW5lX3dlaWdodHMuc3BsaXQoIiwiKSBpZiB4LnN0cmlwKCldOg0KICAgICAgICAgICAgaWYgb3MucGF0aC5pc2ZpbGUocCk6DQogICAgICAgICAgICAgICAgb2ZmbGluZV9wYXRocy5hcHBl")
parts.append("bmQoKHAsIHApKQ0KICAgICAgICAgICAgZWxzZToNCiAgICAgICAgICAgICAgICBwcmludChmIlvmnYPph41dIC0tb2ZmbGluZS13ZWlnaHRzIOaMh+WumueahOaWh+S7tuS4jeWtmOWcqCzot7Pov4c6IHtwfSIsIGZsdXNoPVRydWUpDQogICAgZWxpZiBub3QgYXJn")
parts.append("cy5ub19vZmZsaW5lX3NjYW46DQogICAgICAgIG9mZmxpbmVfcGF0aHMgPSBfc2Nhbl9vZmZsaW5lX3dlaWdodHMoKQ0KICAgICAgICBpZiBvZmZsaW5lX3BhdGhzOg0KICAgICAgICAgICAgcHJpbnQoZiJb5p2D6YeNXSDoh6rliqjmiavmj4/liLAge2xlbihvZmZs")
parts.append("aW5lX3BhdGhzKX0g5Liq56a757q/5p2D6YeN5YCZ6YCJOiIsIGZsdXNoPVRydWUpDQogICAgICAgICAgICBmb3IgcCwgZGVzYyBpbiBvZmZsaW5lX3BhdGhzWzo2XToNCiAgICAgICAgICAgICAgICBwcmludChmIiAge2Rlc2N9IiwgZmx1c2g9VHJ1ZSkNCiAgICAg")
parts.append("ICAgICAgIGlmIGxlbihvZmZsaW5lX3BhdGhzKSA+IDY6DQogICAgICAgICAgICAgICAgcHJpbnQoZiIgIC4uLiDlhbEge2xlbihvZmZsaW5lX3BhdGhzKX0g5LiqIiwgZmx1c2g9VHJ1ZSkNCiAgICAgICAgZWxzZToNCiAgICAgICAgICAgIHByaW50KCJb5p2D6YeN")
parts.append("XSDmnKrlnKjmjILovb3nm67lvZXlj5HnjrDnprvnur/mnYPph40o5Y+v5oyC6L2956S+5Yy65omT5YyF55qEIHJlc25ldDM0IOadg+mHjeaVsOaNrumbhikiLCBmbHVzaD1UcnVlKQ0KICAgICAgICAgICAgX3ByaW50X21vdW50X3RyZWUoIi9rYWdnbGUvaW5wdXQi")
parts.append("LCAi5p2D6YeNIikNCg0KICAgIGdrZiA9IEdyb3VwS0ZvbGQobl9zcGxpdHM9YXJncy5mb2xkcykNCiAgICBncm91cHMgPSB0cmFpbl9jc3ZbIlN0dWR5SW5zdGFuY2VVSUQiXS52YWx1ZXMNCg0KICAgICMg5q+P5Liq5bmz6Z2i54us56uL6LeR5oqYLOiusOW9lSBP")
parts.append("T0Yg5LiOIHRlc3Qg6aKE5rWLDQogICAgcGxhbmVfb29mLCBwbGFuZV90ZXN0LCBwbGFuZV9hdWMsIHBsYW5lX3VpZHMgPSB7fSwge30sIHt9LCB7fQ0KICAgIGZvciBwbGFuZSBpbiBwbGFuZXM6DQogICAgICAgIHByaW50KGYiXG4jIyMjIyMjIyMjIOW5s+mdojog")
parts.append("e3BsYW5lfSAjIyMjIyMjIyMjIikNCiAgICAgICAgb29mID0gbnAuemVyb3MoKGxlbih0cmFpbl9jc3YpLCBsZW4oTEFCRUxTKSkpDQogICAgICAgIGFydGlmYWN0X3Jvb3QgPSBfYXJ0aWZhY3Rfcm9vdChhcmdzKQ0KICAgICAgICBjYW5vbmljYWxfdGVzdF91aWRz")
parts.append("ID0gdGVzdF9kZlsiU3R1ZHlJbnN0YW5jZVVJRCJdLnRvbGlzdCgpDQogICAgICAgIHRyYWluX2ZvbGRfdWlkcyA9IFtdDQogICAgICAgIGZvciBrLCAodHJfaWR4LCB2YV9pZHgpIGluIGVudW1lcmF0ZShna2Yuc3BsaXQodHJhaW5fY3N2LCBncm91cHM9Z3JvdXBz")
parts.append("KSk6DQogICAgICAgICAgICB0cl9zdHVkaWVzID0gdHJhaW5fY3N2Lmlsb2NbdHJfaWR4XVsiU3R1ZHlJbnN0YW5jZVVJRCJdLnRvbGlzdCgpDQogICAgICAgICAgICB2YV9zdHVkaWVzID0gdHJhaW5fY3N2Lmlsb2NbdmFfaWR4XVsiU3R1ZHlJbnN0YW5jZVVJRCJd")
parts.append("LnRvbGlzdCgpDQogICAgICAgICAgICB0cmFpbl9mb2xkX3VpZHMuYXBwZW5kKHRyX3N0dWRpZXMpDQogICAgICAgICAgICBpZiBhcmdzLm9ubHlfZm9sZCA+PSAwIGFuZCBrICE9IGFyZ3Mub25seV9mb2xkOg0KICAgICAgICAgICAgICAgIGNvbnRpbnVlDQogICAg")
parts.append("ICAgICAgICBpZiByZXN1bWVfcm9vdCBhbmQgX2lzX2ZvbGRfZG9uZShhcnRpZmFjdF9yb290LCBrKToNCiAgICAgICAgICAgICAgICBwcmludChmIlthcnRpZmFjdF0gZm9sZCB7a30gZG9uZSwgc2tpcCIsIGZsdXNoPVRydWUpDQogICAgICAgICAgICAgICAgY29u")
parts.append("dGludWUNCiAgICAgICAgICAgIHRyX2RzID0gS25lZURhdGFzZXQodHJfc3R1ZGllcywgbGFiZWxzX2RmLCBkY21fdHJhaW4sIHNlcmllc19kZiwgcGxhbmUsDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG5fY2xpcHM9YXJncy5uX2NsaXBzKQ0KICAg")
parts.append("ICAgICAgICAgdmFfZHMgPSBLbmVlRGF0YXNldCh2YV9zdHVkaWVzLCBsYWJlbHNfZGYsIGRjbV90cmFpbiwgc2VyaWVzX2RmLCBwbGFuZSwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbl9jbGlwcz1hcmdzLm5fY2xpcHMpDQogICAgICAgICAgICBt")
parts.append("b2RlbCA9IEtuZWVNb2RlbChwcmV0cmFpbmVkPWFyZ3MucHJldHJhaW5lZCwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG9mZmxpbmVfcGF0aHM9b2ZmbGluZV9wYXRocywNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGNoYW5uZWxzPWFyZ3Mu")
parts.append("bl9jbGlwcyAqIENMSVApLnRvKGRldmljZSkNCiAgICAgICAgICAgIHByaW50KGYiW+aKmCB7ayArIDF9L3thcmdzLmZvbGRzfV0ge3BsYW5lfSB0cmFpbiB7bGVuKHRyX2RzKX0gLyB2YWwge2xlbih2YV9kcyl9IiwgZmx1c2g9VHJ1ZSkNCiAgICAgICAgICAgIHQw")
parts.append("ID0gdGltZS50aW1lKCkNCiAgICAgICAgICAgIHZhbF9wcmVkLCB2YWxfdWlkID0gcnVuX2ZvbGQobW9kZWwsIHRyX2RzLCB2YV9kcywgZGV2aWNlLCBhcmdzKQ0KICAgICAgICAgICAgZm9sZF90cmFpbl90aW1lID0gdGltZS50aW1lKCkgLSB0MA0KICAgICAgICAg")
parts.append("ICAgZm9yIGosIHVpZCBpbiBlbnVtZXJhdGUodmFsX3VpZCk6DQogICAgICAgICAgICAgICAgcm93ID0gbnAud2hlcmUodHJhaW5fY3N2WyJTdHVkeUluc3RhbmNlVUlEIl0udmFsdWVzID09IHVpZClbMF0NCiAgICAgICAgICAgICAgICBpZiBsZW4ocm93KToNCiAg")
parts.append("ICAgICAgICAgICAgICAgICAgb29mW3Jvd1swXV0gPSB2YWxfcHJlZFtqXQ0KICAgICAgICAgICAgdDAgPSB0aW1lLnRpbWUoKQ0KICAgICAgICAgICAgdGVzdF9wcmVkID0gcHJlZGljdF90ZXN0KG1vZGVsLCB0ZXN0X2RmLCB0ZXN0X3NlcmllcywgZGNtX3Rlc3Qs")
parts.append("IGRldmljZSwgcGxhbmUsIG5fY2xpcHM9YXJncy5uX2NsaXBzKQ0KICAgICAgICAgICAgZm9sZF9pbmZlcmVuY2VfdGltZSA9IHRpbWUudGltZSgpIC0gdDANCiAgICAgICAgICAgIG1ldGEgPSB7DQogICAgICAgICAgICAgICAgImZvbGQiOiBrLA0KICAgICAgICAg")
parts.append("ICAgICAgICJwbGFuZSI6IHBsYW5lLA0KICAgICAgICAgICAgICAgICJuX2NsaXBzIjogYXJncy5uX2NsaXBzLA0KICAgICAgICAgICAgICAgICJpbWFnZV9zaXplIjogSU1HX1NJWkUsDQogICAgICAgICAgICAgICAgImVwb2NocyI6IGFyZ3MuZXBvY2hzLA0KICAg")
parts.append("ICAgICAgICAgICAgICJiYXRjaF9zaXplIjogYXJncy5iYXRjaCwNCiAgICAgICAgICAgICAgICAic2VlZCI6IGFyZ3Muc2VlZCwNCiAgICAgICAgICAgICAgICAiZ2l0X2NvbW1pdCI6IF9fZ2l0X2NvbW1pdCgpLA0KICAgICAgICAgICAgICAgICJmb2xkX3RyYWlu")
parts.append("X3RpbWVfc2VjIjogZm9sZF90cmFpbl90aW1lLA0KICAgICAgICAgICAgICAgICJmb2xkX2luZmVyZW5jZV90aW1lX3NlYyI6IGZvbGRfaW5mZXJlbmNlX3RpbWUsDQogICAgICAgICAgICB9DQogICAgICAgICAgICBfc2F2ZV9mb2xkX2FydGlmYWN0KGFydGlmYWN0")
parts.append("X3Jvb3QsIGssIHRlc3RfcHJlZCwgY2Fub25pY2FsX3Rlc3RfdWlkcywgbWV0YSkNCiAgICAgICAgICAgIHRyeToNCiAgICAgICAgICAgICAgICB3ZGlyID0gYXJ0aWZhY3Rfcm9vdCAvIGYiZm9sZHtrfSIgLyBwbGFuZQ0KICAgICAgICAgICAgICAgIHdkaXIubWtk")
parts.append("aXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQ0KICAgICAgICAgICAgICAgIHdwYXRoID0gd2RpciAvICJtb2RlbC5wdCINCiAgICAgICAgICAgICAgICB0b3JjaC5zYXZlKG1vZGVsLnN0YXRlX2RpY3QoKSwgc3RyKHdwYXRoKSkNCiAgICAgICAgICAgICAg")
parts.append("ICBtZXRhMiA9IGRpY3QobWV0YSkNCiAgICAgICAgICAgICAgICBtZXRhMlsiY2hhbm5lbHMiXSA9IGludChhcmdzLm5fY2xpcHMgKiBDTElQKQ0KICAgICAgICAgICAgICAgIG1ldGEyWyJudW1fbGFiZWxzIl0gPSBsZW4oTEFCRUxTKQ0KICAgICAgICAgICAgICAg")
parts.append("ICh3ZGlyIC8gIm1vZGVsX21ldGEuanNvbiIpLndyaXRlX3RleHQoDQogICAgICAgICAgICAgICAgICAgIGpzb24uZHVtcHMobWV0YTIsIGVuc3VyZV9hc2NpaT1GYWxzZSwgaW5kZW50PTIpLCBlbmNvZGluZz0idXRmLTgiKQ0KICAgICAgICAgICAgICAgIHByaW50")
parts.append("KGYiW3dlaWdodHNdIHNhdmVkIHt3cGF0aH0iLCBmbHVzaD1UcnVlKQ0KICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOg0KICAgICAgICAgICAgICAgIHByaW50KGYiW3dlaWdodHNdIHNhdmUgZmFpbGVkOiB7ZX0iLCBmbHVzaD1UcnVlKQ0KICAgICAg")
parts.append("ICAgICAgcHJpbnQoZiJbYXJ0aWZhY3RdIHNhdmVkIGZvbGR7a30gLT4ge2FydGlmYWN0X3Jvb3QgLyBmJ2ZvbGR7a30nfSIsIGZsdXNoPVRydWUpDQogICAgICAgICMgZXhwZXJpbWVudC1sZXZlbCBtZXRhIG9uY2UgcGVyIHBsYW5lDQogICAgICAgIF9zYXZlX2V4")
parts.append("cGVyaW1lbnRfbWV0YShhcnRpZmFjdF9yb290LCB0cmFpbl9mb2xkX3VpZHMsIGNhbm9uaWNhbF90ZXN0X3VpZHMsIGFyZ3MpDQogICAgICAgIHByaW50KGYiW2FydGlmYWN0XSBzYXZlZCBleHBlcmltZW50IG1ldGEgLT4ge2FydGlmYWN0X3Jvb3R9IiwgZmx1c2g9")
parts.append("VHJ1ZSkNCiAgICAgICAgIyDor6XlubPpnaIgT09GIEFVQyjlro/lubPlnYcpDQogICAgICAgIHVpZF9saXN0ID0gdHJhaW5fY3N2WyJTdHVkeUluc3RhbmNlVUlEIl0udG9saXN0KCkNCiAgICAgICAgaWYgYXJncy5vbmx5X2ZvbGQgPj0gMDoNCiAgICAgICAgICAg")
parts.append("IHByaW50KGYiW3twbGFuZX1dIOaOoui3r+aooeW8jygtLW9ubHktZm9sZCB7YXJncy5vbmx5X2ZvbGR9KTpPT0Yg5Y2V5oqY5LiN5a6M5pW0LCINCiAgICAgICAgICAgICAgICAgIGYi6Lez6L+HIE9PRiBBVUMo5p2D6YeN6K6wIDA95Z2H562JKTt0ZXN0IOaOqOeQ")
parts.append("hueUqOivpeaKmOaooeWeiyIpDQogICAgICAgICAgICBhdWMgPSAwLjANCiAgICAgICAgZWxzZToNCiAgICAgICAgICAgIGF1YyA9IHBsYW5lX29vZl9hdWMob29mLCB0cmFpbl9jc3ZbWyJTdHVkeUluc3RhbmNlVUlEIl0gKyBMQUJFTFNdLCB1aWRfbGlzdCkNCiAg")
parts.append("ICAgICAgICAgIHByaW50KGYiW3twbGFuZX1dIE9PRiDlro/lubPlnYcgQVVDIHthdWM6LjRmfSIpDQogICAgICAgIHBsYW5lX29vZltwbGFuZV0sIHBsYW5lX2F1Y1twbGFuZV0sIHBsYW5lX3VpZHNbcGxhbmVdID0gb29mLCBhdWMsIHVpZF9saXN0DQogICAgICAg")
parts.append("ICMgbG9hZCBzYXZlZCBmb2xkIHByZWRpY3Rpb25zIGZvciB0aGlzIHBsYW5lDQogICAgICAgIHBlcl9mb2xkID0gW10NCiAgICAgICAgbWlzc2luZyA9IFtdDQogICAgICAgIGZvciBrIGluIHJhbmdlKGFyZ3MuZm9sZHMpOg0KICAgICAgICAgICAgcCA9IGFydGlm")
parts.append("YWN0X3Jvb3QgLyBmImZvbGR7a30iIC8gInRlc3RfcHJlZC5ucHkiDQogICAgICAgICAgICBpZiBub3QgcC5leGlzdHMoKToNCiAgICAgICAgICAgICAgICBtaXNzaW5nLmFwcGVuZChrKQ0KICAgICAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgICAgICB0cnk6")
parts.append("DQogICAgICAgICAgICAgICAgcGVyX2ZvbGQuYXBwZW5kKF9sb2FkX2ZvbGRfcHJlZGljdGlvbnMoYXJ0aWZhY3Rfcm9vdCwgaywgY2Fub25pY2FsX3Rlc3RfdWlkcykpDQogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6DQogICAgICAgICAgICAgICAg")
parts.append("bWlzc2luZy5hcHBlbmQoaykNCiAgICAgICAgICAgICAgICBwcmludChmIltlbnNlbWJsZV0gZm9sZHtrfSBpbnZhbGlkOiB7ZX0iLCBmbHVzaD1UcnVlKQ0KICAgICAgICBpZiBsZW4obWlzc2luZykgPT0gYXJncy5mb2xkczoNCiAgICAgICAgICAgIHN5cy5leGl0")
parts.append("KGYiW2Vuc2VtYmxlXSBubyB2YWxpZCBmb2xkIHByZWRpY3Rpb25zIGF2YWlsYWJsZSB1bmRlciB7YXJ0aWZhY3Rfcm9vdH0iKQ0KICAgICAgICBpZiBtaXNzaW5nOg0KICAgICAgICAgICAgcHJpbnQoZiJbZW5zZW1ibGVdIG1pc3NpbmcgZm9sZHM6IHttaXNzaW5n")
parts.append("fTsgZW5zZW1ibGUgd2l0aCBhdmFpbGFibGUgZm9sZHMgb25seSIsIGZsdXNoPVRydWUpDQogICAgICAgIHBsYW5lX3Rlc3RbcGxhbmVdID0gbnAubWVhbihwZXJfZm9sZCwgYXhpcz0wKSBpZiBwZXJfZm9sZCBlbHNlIE5vbmUNCiAgICAgICAgaWYgcGxhbmVfdGVz")
parts.append("dFtwbGFuZV0gaXMgbm90IE5vbmU6DQogICAgICAgICAgICBwcmludChmIltlbnNlbWJsZV0ge3BsYW5lfSBlbnNlbWJsZWQgZnJvbSB7bGVuKHBlcl9mb2xkKX0gZm9sZHMsIHNoYXBlPXtwbGFuZV90ZXN0W3BsYW5lXS5zaGFwZX0iKQ0KDQogICAgIyDono3lkIgN")
parts.append("CiAgICBpZiBhcmdzLmZ1c2UgPT0gIndlaWdodCI6DQogICAgICAgIHdzdW0gPSBzdW0obWF4KGEsIDFlLTYpIGZvciBhIGluIHBsYW5lX2F1Yy52YWx1ZXMoKSkNCiAgICAgICAgd2VpZ2h0cyA9IHtwOiBtYXgoYSwgMWUtNikgLyB3c3VtIGZvciBwLCBhIGluIHBs")
parts.append("YW5lX2F1Yy5pdGVtcygpfQ0KICAgICAgICBwcmludChmIlvono3lkIhdIE9PRiBBVUMg5Yqg5p2DOiB7IHtwOiByb3VuZCh3LCA0KSBmb3IgcCwgdyBpbiB3ZWlnaHRzLml0ZW1zKCl9IH0iKQ0KICAgIGVsc2U6DQogICAgICAgIHdlaWdodHMgPSB7cDogMS4wIC8g")
parts.append("bGVuKHBsYW5lcykgZm9yIHAgaW4gcGxhbmVzfQ0KICAgICAgICBwcmludCgiW+iejeWQiF0g566A5Y2V5bmz5Z2HIikNCiAgICBmdXNlZCA9IG5wLnplcm9zX2xpa2UocGxhbmVfdGVzdFtwbGFuZXNbMF1dKQ0KICAgIGZvciBwLCB3IGluIHdlaWdodHMuaXRlbXMo")
parts.append("KToNCiAgICAgICAgZnVzZWQgKz0gdyAqIHBsYW5lX3Rlc3RbcF0NCiAgICBzdWIgPSBwZC5EYXRhRnJhbWUoeyJTdHVkeUluc3RhbmNlVUlEIjogdGVzdF9kZlsiU3R1ZHlJbnN0YW5jZVVJRCJdLCAqKnsNCiAgICAgICAgTEFCRUxTW2pdOiBmdXNlZFs6LCBqXSBm")
parts.append("b3IgaiBpbiByYW5nZShsZW4oTEFCRUxTKSl9fSkNCiAgICBzdWJfcGF0aCA9IG9zLnBhdGguam9pbihvdXQsICJzdWJtaXNzaW9uLmNzdiIpDQogICAgc3ViLnRvX2NzdihzdWJfcGF0aCwgaW5kZXg9RmFsc2UpDQogICAgcHJpbnQoZiJcbuaPkOS6pOaWh+S7tuW3")
parts.append("sueUn+aIkDoge3N1Yl9wYXRofSAgc2hhcGU9e3N1Yi5zaGFwZX0iKQ0KDQoNCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6DQogICAgbWFpbigpDQo=")
open('/kaggle/working/rsna_v3.py','wb').write(base64.b64decode(''.join(parts)))
print('wrote rsna_v3.py', os.path.getsize('/kaggle/working/rsna_v3.py'))

import base64, os
# embed exp012_llm_primary.py
parts=[]
parts.append("IiIiRVhQLTAxMiBsYWJlbCBidWlsZGVyOiBMTE0tcHJpbWFyeSBzb2Z0IGxhYmVscyB3aXRoIHNpbGVuY2UgbWFza2luZy4KCldoeTogcmVwb3J0LXJ1bGVzIGtleSDiiYggMC44MSB2cyBnb2xkOyBzdGV2ZW4gdjRfYmxlbmQga2V5IOKJiCAwLjg5LgpFWFAtMDA5")
parts.append("IGZpbGxlZCBzaWxlbmNlIDAuMjUgYXMgY29uZmlkZW50IG5lZ2F0aXZlcyAoMC41ODYpLgpFWFAtMDEwIG9ubHkgZmlsbGVkIHJlbWFpbmluZyBnYXBzIHNvIHJlcG9ydCBydWxlcyBzdGlsbCBkb21pbmF0ZWQgKDAuNjAxKS4KRVhQLTAxMiB1c2VzIExMTSBhcyBQ")
parts.append("UklNQVJZIGZvciBub24tZ29sZCwgc2lsZW5jZSDihpIgTmFOLCBnb2xkIG92ZXJyaWRlcy4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucApp")
parts.append("bXBvcnQgcGFuZGFzIGFzIHBkCgpMQUJFTFMgPSBbCiAgICAiQUNMIiwgIk1DTCIsICJNZWRpYWwgTWVuaXNjdXMiLCAiTGF0ZXJhbCBNZW5pc2N1cyIsCiAgICAiTWVkaWFsIE9BIiwgIkxhdGVyYWwgT0EiLCAiUEYgT0EiLCAiRWZmdXNpb24iLAogICAgIlN5bm92")
parts.append("aXRpcyIsICJCYWtlcidzIiwgIkNvbnR1c2lvbiIsICJGcmFjdHVyZSIsCl0KCiMgc2lsZW5jZSAvIGRlYWQgYnVja2V0cyBpbiBzdGV2ZW4gQ1NWcyDigJQgbmV2ZXIgdHJhaW4gb24gdGhlc2UKU0lMRU5DRV9ST1VOREVEID0gezAuMjUsIDAuNX0KCiMgbWF0Y2gg")
parts.append("cmVwb3J0LWxhYmVsIGNhbGlicmF0aW9uIChwb3MgMC43Mi0wLjk1IC8gbmVnIDAuMDQtMC4yMCkKQ0xJUF9MTywgQ0xJUF9ISSA9IDAuMDgsIDAuOTIKCgpkZWYgX2ZpbmRfbGxtKCkgLT4gUGF0aDoKICAgIGVudiA9IG9zLmVudmlyb24uZ2V0KCJFWFAwMTJfTExN")
parts.append("X0NTViIpCiAgICBpZiBlbnYgYW5kIFBhdGgoZW52KS5pc19maWxlKCk6CiAgICAgICAgcmV0dXJuIFBhdGgoZW52KQogICAgY2FuZHMgPSBbCiAgICAgICAgUGF0aChyIkQ6L+ahjOmdoi9LYWdnbGUvcnNuYV9rbmVlL19kYXRhc2V0X3N0YWdlL3N0ZXZlbl9sYWJl")
parts.append("bHMvbGxtX2xhYmVsc192NF9ibGVuZC5jc3YiKSwKICAgICAgICBQYXRoKHIiRDov5qGM6Z2iL0thZ2dsZS9yc25hX2tuZWUvX2RhdGFzZXRfc3RhZ2Uvc3RldmVuX2xhYmVscy9sbG1fbGFiZWxzX3YyLmNzdiIpLAogICAgXQogICAgZm9yIHJvb3QgaW4gKCIva2Fn")
parts.append("Z2xlL2lucHV0IiwgciJEOi/moYzpnaIvS2FnZ2xlL3JzbmFfa25lZS9fZGF0YXNldF9zdGFnZSIpOgogICAgICAgIGlmIG5vdCBQYXRoKHJvb3QpLmlzX2RpcigpOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGZvciBwIGluIFBhdGgocm9vdCkucmdsb2Io")
parts.append("ImxsbV9sYWJlbHNfdjRfYmxlbmQuY3N2Iik6CiAgICAgICAgICAgIHJldHVybiBwCiAgICAgICAgZm9yIHAgaW4gUGF0aChyb290KS5yZ2xvYigibGxtX2xhYmVsc192Mi5jc3YiKToKICAgICAgICAgICAgcmV0dXJuIHAKICAgIGZvciBwIGluIGNhbmRzOgogICAg")
parts.append("ICAgIGlmIHAuaXNfZmlsZSgpOgogICAgICAgICAgICByZXR1cm4gcAogICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoImxsbV9sYWJlbHNfdjRfYmxlbmQuY3N2IG5vdCBmb3VuZCIpCgoKZGVmIGNsZWFuX2xsbV92YWx1ZXMoYXJyOiBucC5uZGFycmF5KSAtPiBu")
parts.append("cC5uZGFycmF5OgogICAgb3V0ID0gYXJyLmFzdHlwZShucC5mbG9hdDY0KS5jb3B5KCkKICAgIG91dCA9IG5wLndoZXJlKG5wLmlzaW4obnAucm91bmQob3V0LCAyKSwgbGlzdChTSUxFTkNFX1JPVU5ERUQpKSwgbnAubmFuLCBvdXQpCiAgICAjIHNocmluayBleHRy")
parts.append("ZW1lcyBzbyBCQ0UgZG9lcyBub3Qgb3ZlcmZpdCAwLjAyLzAuOTggcmVwb3J0LXN0eWxlIG5vaXNlCiAgICBvdXQgPSAwLjUgKyAob3V0IC0gMC41KSAqIDAuNzUKICAgIG91dCA9IG5wLmNsaXAob3V0LCBDTElQX0xPLCBDTElQX0hJKQogICAgcmV0dXJuIG91dAoK")
parts.append("CmRlZiBidWlsZF9leHAwMTJfbGFiZWxzKHRyYWluX2Nzdjogc3RyIHwgUGF0aCwgdmVyYm9zZTogYm9vbCA9IFRydWUpIC0+IHBkLkRhdGFGcmFtZToKICAgIHRyYWluID0gcGQucmVhZF9jc3YodHJhaW5fY3N2KQogICAgZGYgPSB0cmFpbltbIlN0dWR5SW5zdGFu")
parts.append("Y2VVSUQiXSArIExBQkVMU10uY29weSgpCiAgICBnb2xkX2NudCA9IHtjOiBpbnQoZGZbY10ubm90bmEoKS5zdW0oKSkgZm9yIGMgaW4gTEFCRUxTfQoKICAgIGxsbV9wYXRoID0gX2ZpbmRfbGxtKCkKICAgIGxsbSA9IHBkLnJlYWRfY3N2KGxsbV9wYXRoKQogICAg")
parts.append("bGxtID0gbGxtLnNldF9pbmRleCgiU3R1ZHlJbnN0YW5jZVVJRCIpCiAgICBpZiB2ZXJib3NlOgogICAgICAgIHByaW50KGYiW0VYUC0wMTJdIExMTSBzb3VyY2U6IHtsbG1fcGF0aC5uYW1lfSByb3dzPXtsZW4obGxtKX0iKQoKICAgIG5fZmlsbCA9IDAKICAgIGZv")
parts.append("ciBjIGluIExBQkVMUzoKICAgICAgICBpZiBjIG5vdCBpbiBsbG0uY29sdW1uczoKICAgICAgICAgICAgaWYgdmVyYm9zZToKICAgICAgICAgICAgICAgIHByaW50KGYiICB7YzoxOHN9IExMTSBtaXNzaW5nIGNvbHVtbiIpCiAgICAgICAgICAgIGNvbnRpbnVlCiAg")
parts.append("ICAgICAgbWFzayA9IGRmW2NdLmlzbmEoKS52YWx1ZXMKICAgICAgICB1aWRzID0gZGYubG9jW21hc2ssICJTdHVkeUluc3RhbmNlVUlEIl0KICAgICAgICByYXcgPSBsbG1bY10ucmVpbmRleCh1aWRzKS50b19udW1weShkdHlwZT1ucC5mbG9hdDY0KQogICAgICAg")
parts.append("IGNsZWFuZWQgPSBjbGVhbl9sbG1fdmFsdWVzKHJhdykKICAgICAgICBkZi5sb2NbbWFzaywgY10gPSBjbGVhbmVkCiAgICAgICAgbl91c2UgPSBpbnQobnAuaXNmaW5pdGUoY2xlYW5lZCkuc3VtKCkpCiAgICAgICAgbl9zaWwgPSBpbnQobnAuaXNmaW5pdGUocmF3")
parts.append("KS5zdW0oKSkgLSBuX3VzZQogICAgICAgIG5fZmlsbCArPSBuX3VzZQogICAgICAgIGlmIHZlcmJvc2U6CiAgICAgICAgICAgIHByaW50KAogICAgICAgICAgICAgICAgZiIgIHtjOjE4c30gZ29sZCB7Z29sZF9jbnRbY106NGR9ICsgTExNIHtuX3VzZTo0ZH0iCiAg")
parts.append("ICAgICAgICAgICAgICBmIiAoc2lsZW5jZS9kcm9wIHtuX3NpbDo0ZH0pID0ge2ludChkZltjXS5ub3RuYSgpLnN1bSgpKTo1ZH0iCiAgICAgICAgICAgICkKICAgIGlmIHZlcmJvc2U6CiAgICAgICAgcHJpbnQoZiJbRVhQLTAxMl0gdG90YWwgTExNIGNlbGxzIGZp")
parts.append("bGxlZDoge25fZmlsbH0iKQogICAgcmV0dXJuIGRmCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHJvb3QgPSBQYXRoKG9zLmVudmlyb24uZ2V0KCJSU05BX1JPT1QiLCByIkQ6L+ahjOmdoi9LYWdnbGUvcnNuYV9rbmVlIikpCiAgICBvdXQgPSBidWls")
parts.append("ZF9leHAwMTJfbGFiZWxzKHJvb3QgLyAidHJhaW4uY3N2IikKICAgIGRzdCA9IHJvb3QgLyAibGFiZWxzIiAvICJ0cmFpbl9leHAwMTJfbGxtX3ByaW1hcnkuY3N2IgogICAgZHN0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBv")
parts.append("dXQudG9fY3N2KGRzdCwgaW5kZXg9RmFsc2UpCiAgICBwcmludCgid3JvdGUiLCBkc3QsIG91dC5zaGFwZSkK")
open('/kaggle/working/exp012_llm_primary.py','wb').write(base64.b64decode(''.join(parts)))
print('wrote exp012_llm_primary.py', os.path.getsize('/kaggle/working/exp012_llm_primary.py'))
open('/kaggle/working/rsna_report_labels.py','w').write('# stub; EXP-012 uses LLM primary\n')
print('embeds ready')


In [ ]:
import os, glob
hits = glob.glob('/kaggle/input/**/llm_labels_v4_blend.csv', recursive=True)
if not hits:
    hits = glob.glob('/kaggle/input/**/llm_labels*.csv', recursive=True)
print('llm csv hits', hits)
assert hits, 'FAIL: steven LLM labels dataset not mounted'
os.environ['EXP012_LLM_PRIMARY'] = '1'
os.environ['EXP012_LLM_CSV'] = hits[0]
print('EXP012_LLM_PRIMARY=1 csv=', hits[0])


In [ ]:
import subprocess, sys, os, json
os.chdir('/kaggle/working')
os.environ['EXP012_LLM_PRIMARY'] = '1'
cmd = [sys.executable, '-u', 'rsna_v3.py',
       '--planes', 'axial',
       '--epochs', '3', '--folds', '5', '--only-fold', '0',
       '--batch', '8', '--n-clips', '2',
       '--experiment-id', 'exp_012_llm_primary_axial_fold0',
       '--verbose']
print('=== EXP-012 LLM-primary + Axial fold0 ===', flush=True)
print(' '.join(cmd), flush=True)
r = subprocess.run(cmd)
print('returncode', r.returncode, flush=True)

import numpy as np, pandas as pd
artifact_root = '/kaggle/working/artifacts/exp_012_llm_primary_axial_fold0'
canon = None
for uid_path in [
    os.path.join(artifact_root, 'test_uid.csv'),
    os.path.join(artifact_root, 'fold0', 'Axial', 'test_uid.csv'),
    os.path.join(artifact_root, 'fold0', 'test_uid.csv'),
]:
    if os.path.isfile(uid_path):
        canon = pd.read_csv(uid_path)['StudyInstanceUID'].tolist()
        print('canonical uids', len(canon), uid_path)
        break
for d in [
    os.path.join(artifact_root, 'fold0', 'Axial'),
    os.path.join(artifact_root, 'fold0'),
]:
    ok = all(os.path.isfile(os.path.join(d, n)) for n in ['test_pred.npy', 'test_uid.csv', 'meta.json'])
    print(f'[artifact] {d}: {"PASS" if ok else "MISSING"}')
    if ok:
        meta = json.load(open(os.path.join(d, 'meta.json'), encoding='utf-8'))
        pred = np.load(os.path.join(d, 'test_pred.npy'))
        print(' meta', {k: meta.get(k) for k in ['fold','plane','completion_status','fold_train_time_sec','prediction_shape']})
        print(' pred', pred.shape, 'finite', bool(np.isfinite(pred).all()),
              'range', float(pred.min()), float(pred.max()), 'mean', float(pred.mean()))
        assert meta.get('completion_status')=='done'
sub = '/kaggle/working/submission.csv'
if os.path.isfile(sub):
    s = pd.read_csv(sub)
    print('[sub]', s.shape, 'NaN', int(s.isna().sum().sum()))
print('[exp012] done')
